# 02 — Figures & analyses (version R1, corrigée)

Copie de `Pipeline_analyse/02_Figures_Analyses.ipynb` (conservé intact), branchée sur `outputs/` de ce dossier (produits par `01_Data_Cleaning_Preparation_R1.ipynb`). Corrections signalées par `# R1-FIX` :

| # | Section | Correction |
|---|---|---|
| 6 | §6 Régressions logistiques à l'admission | Ajout de la constante ; retrait de `death` (critère de jugement utilisé à tort comme covariable) ; brûlure thermique = référence. |
| 7 | §7 Cox temps-dépendant | Les points de coupure postérieurs à la fin de suivi (et `inf`) créaient des lignes *après* l'événement du patient. Corrigé ; événements > J90 censurés à J90. |
| 9 | §14 Figure 2 | Les patients sans natrémie exploitable ne sont plus classés « normonatrémiques ». |
| 10 | §11 Supp Fig 6 | Remplissage vasculaire : 4 valeurs négatives traitées comme manquantes. |
| 11 | §13 Clustering | Haptoglobine retirée des variables (79 % de données manquantes). |
| 12 | §9 Supp Fig 2 | Pourcentages de sévérité calculés sur toute la cohorte (n = 1 758), comme l'incidence. |
| 8 | §10-11 Épisodes | « Diurétiques avant l'épisode » = au moins une administration la veille (le code comptait *exactement une*). |

Figures enregistrées dans `figures/`.

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')   # avertissements de depreciation (pandas/seaborn) sans effet sur les resultats

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from matplotlib_venn import venn2
import matplotlib.patheffects as patheffects
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, fisher_exact
import statsmodels.api as sm
from tableone import TableOne
from lifelines import CoxTimeVaryingFitter

mpl.rcParams['font.family'] = 'DejaVu Serif'
sns.set_style("white")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
np.random.seed(42)


## 1. Chargement des donnees

**Chemins relatifs** : ce notebook doit etre place a la racine du dossier
pipeline, a cote du dossier `outputs/` genere par
`01_Data_Cleaning_Preparation.ipynb` (executer ce dernier avant celui-ci).


In [2]:

DATA_FOLDER = "outputs/"

CRF  = pd.read_excel(DATA_FOLDER + "dffinal_clean.xlsx")      # 1 ligne = 1 patient (n=1758)
CRF3 = pd.read_excel(DATA_FOLDER + "hyper_episode_full.xlsx")  # 1 ligne = 1 episode d'hypernatremie
CRF2 = pd.read_excel(DATA_FOLDER + "hypo_episode_full.xlsx")   # 1 ligne = 1 episode d'hyponatremie

print(f"Patients (CRF)               : {len(CRF)}")
print(f"Episodes hypernatremie (CRF3) : {len(CRF3)}")
print(f"Episodes hyponatremie (CRF2)  : {len(CRF2)}")

print()
print("AKI_KDIGO, n (%)      :", int(CRF['AKI_KDIGO'].sum()), f"({100*CRF['AKI_KDIGO'].mean():.1f}%)")
print("AKI_stage distribution :")
print(CRF['AKI_stage'].value_counts(dropna=False).sort_index())
print("MAKE28 :", int(CRF['MAKE28'].sum()))
print("MAKE90 :", int(CRF['MAKE90'].sum()))


Patients (CRF)               : 1758
Episodes hypernatremie (CRF3) : 2206
Episodes hyponatremie (CRF2)  : 3274

AKI_KDIGO, n (%)      : 404 (23.0%)
AKI_stage distribution :
AKI_stage
0    1354
1     179
2      73
3     152
Name: count, dtype: int64
MAKE28 : 200
MAKE90 : 207


## Supplementary Figure 1 — Flow chart of patients

Regenerated 2026-07-20. Screening: 2090 admitted − 332 non-burned = 1758 (final cohort = `len(CRF)`). The <18 exclusion box was removed (cohort keeps adolescents, per author decision).

In [3]:
# Supplementary Figure 1 - Flow chart of patients (corrected: only non-burned exclusion; 2090 -> -332 -> 1758)
# R1-FIX 2026-09-30 : separateur de milliers (2,090 / 1,758) comme dans le texte du manuscrit
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
plt.rcParams.update({"font.family":"DejaVu Serif"})
fig,ax=plt.subplots(figsize=(8.2,7.0)); ax.set_xlim(0,100); ax.set_ylim(0,100); ax.axis("off")
def box(cx,cy,w,h,title,n):
    ax.add_patch(Rectangle((cx-w/2,cy-h/2),w,h,fill=True,facecolor="white",edgecolor="black",linewidth=1.6))
    ax.text(cx,cy+h*0.16,title,ha="center",va="center",fontsize=15)
    ax.text(cx,cy-h*0.20,f"N = {n:,}",ha="center",va="center",fontsize=17,fontweight="bold")
def line(x1,y1,x2,y2): ax.plot([x1,x2],[y1,y2],color="black",lw=1.5)
# boxes
box(34,84,60,20,"Patients admitted to ICU Burn Center",2090)
box(34,18,60,20,"Patients admitted for burn",1758)
# vertical connector
line(34,74,34,28)
# exclusion branch (non burned only)
line(34,52,63,52)
ax.text(82,52,"Non-burned patients",ha="center",va="center",fontsize=15)
ax.text(82,46,"N = 332",ha="center",va="center",fontsize=16,fontweight="bold")
plt.subplots_adjust(left=0.01,right=0.99,top=0.99,bottom=0.01)
fig.savefig("figures/suppfig1_flowchart.png",dpi=200,bbox_inches="tight",facecolor="white")
from PIL import Image; print("saved",Image.open("figures/suppfig1_flowchart.png").size)
plt.show()

saved (1647, 1412)


**Historique -- bug `creat_baseline` (corrige a la source, Notebook 1)** :
la fonction `creat_baseline()` du pipeline original calculait correctement la
creatinine de base, mais son `return` renvoyait `eGFR_adm` a la place
(`creat_baseline` etait une copie exacte de `eGFR_adm`, correlation r=1.0
confirmee). Un second bug (unites incoherentes mg/dL vs µmol/L) empechait en
plus le stade KDIGO 2 d'apparaitre meme apres correction du premier. Ces deux
bugs affectaient `AKI_KDIGO`, `AKI_stage`, `MAKE28` et `MAKE90`. Les deux sont
desormais corriges directement dans `01_Data_Cleaning_Preparation.ipynb`
(sections 5 et 6) -- `CRF` ci-dessus contient donc deja les valeurs
corrigees, sans patch a appliquer ici.

**Note de validation (2026-07-07)** : en reconstruisant `AKI_KDIGO`/`AKI_stage`
depuis les 39 fichiers bruts 2012-2023, ce pipeline "propre" retrouve
`MAKE28`/`MAKE90` a l'identique des chiffres deja publies dans le manuscrit
(187/199), et `AKI_KDIGO` a 3 patients pres sur 1758 (371 vs 374, 21.1% vs
21.3% -- meme repartition de stades 2/3, ecart uniquement entre stades 0/1).
Cet ecart residuel provient tres probablement d'une difference mineure entre
le fichier de creatinines deja nettoye utilise pour les chiffres publies et
une relecture stricte des fichiers bruts ici -- a confirmer avec Louis avant
de considerer ce chiffre comme la nouvelle reference.


## 2. Definitions communes

Calculees **une seule fois** ici et reutilisees dans tout le notebook.

- `Hyponatremia` / `Hypernatremia` : au moins un episode pendant le sejour
  (base sur `min_natremia_value` / `max_natremia_value`).
- `Dysnatremia` : hypo OU hyper.
- `MAKE_at_discharge` : **MAKE90 corrige** (voir Section 1bis), definition
  utilisee pour le Tableau 1, l'Abstract et le modele de Cox du papier.
- `AKI` : **AKI_KDIGO corrige** (voir Section 1bis), calculee a partir des
  dosages seriés de creatinine, en appliquant les vrais criteres KDIGO
  (ratio creat/baseline >= 1.5/2/3, ou +0.3 mg/dL en 48h, ou RRT).

**Note de nettoyage (decisions de Louis Boutin, 2026-07-07)** :

- `IRA` (n=711, champ CRF brut renseigne manuellement) vs `AKI_KDIGO`
  (n=374 apres correction du bug ci-dessus, calculee de facon standardisee
  et reproductible a partir des creatinines seriees) : les deux ne
  concordent pas parfaitement (voir ci-dessous), ce qui est attendu -- `IRA`
  peut refleter un jugement clinique plus large que le seul critere
  creatinine. Seule `AKI_KDIGO` est utilisee dans les figures/tableaux de ce
  notebook.
- `MAKE28 OU MAKE90` vs `MAKE90` seul : MAKE28 et MAKE90 sont deux criteres
  composites **independants** (RRT ou deces ou creat/baseline >=1.5, evalue
  respectivement a J28 et J90) — un patient peut se degrader a J28 puis
  recuperer d'ici J90, donc MAKE28=Vrai n'implique pas MAKE90=Vrai. Seul
  `MAKE90` (`MAKE_at_discharge`) est utilise dans les figures/tableaux de ce
  notebook.


In [4]:

CRF['Hyponatremia']  = (CRF['min_natremia_value'] < 135).astype(int)
CRF['Hypernatremia'] = (CRF['max_natremia_value'] > 145).astype(int)
CRF['Dysnatremia']   = ((CRF['Hyponatremia'] == 1) | (CRF['Hypernatremia'] == 1)).astype(int)
CRF['Both']          = ((CRF['Hyponatremia'] == 1) & (CRF['Hypernatremia'] == 1)).astype(int)

def is_true(x):
    if pd.isna(x):
        return False
    if isinstance(x, str):
        return x.strip().lower() in ['vrai', 'true', '1', 'yes', 'oui']
    return bool(x)

CRF['MAKE_at_discharge'] = CRF['MAKE90'].apply(is_true).astype(int)
CRF['Death']             = CRF['death'].apply(is_true).astype(int) if CRF['death'].dtype == object else CRF['death'].astype(int)

n_total = len(CRF)
print("N total                          :", n_total)
print("Hyponatremia, n (%)               :", CRF['Hyponatremia'].sum(), f"({100*CRF['Hyponatremia'].mean():.1f}%)")
print("Hypernatremia, n (%)              :", CRF['Hypernatremia'].sum(), f"({100*CRF['Hypernatremia'].mean():.1f}%)")
print("Dysnatremia, n (%)                :", CRF['Dysnatremia'].sum(), f"({100*CRF['Dysnatremia'].mean():.1f}%)")
print("MAKE_at_discharge (MAKE90), n (%) :", CRF['MAKE_at_discharge'].sum(), f"({100*CRF['MAKE_at_discharge'].mean():.1f}%)")
print("AKI_KDIGO, n (%)                  :", int(CRF['AKI_KDIGO'].sum()), f"({100*CRF['AKI_KDIGO'].mean():.1f}%)")

if 'IRA' in CRF.columns:
    agree = (CRF['IRA'].fillna(0).astype(int) == CRF['AKI_KDIGO'].fillna(0).astype(int)).sum()
    print()
    print("Pour memoire : IRA (champ CRF brut, n=%d) et AKI_KDIGO (corrigee, n=%d) concordent sur %d/%d lignes."
          % (CRF['IRA'].fillna(0).astype(int).sum(), int(CRF['AKI_KDIGO'].sum()), agree, len(CRF)))
    print("Seule AKI_KDIGO est utilisee ci-dessous.")


N total                          : 1758
Hyponatremia, n (%)               : 888 (50.5%)
Hypernatremia, n (%)              : 418 (23.8%)
Dysnatremia, n (%)                : 990 (56.3%)
MAKE_at_discharge (MAKE90), n (%) : 207 (11.8%)
AKI_KDIGO, n (%)                  : 404 (23.0%)

Pour memoire : IRA (champ CRF brut, n=711) et AKI_KDIGO (corrigee, n=404) concordent sur 1393/1758 lignes.
Seule AKI_KDIGO est utilisee ci-dessous.


## 3. Table 1 — Caracteristiques de la population

Reconstruit a partir de `tableau_1_post_these.ipynb` (version finale, utilisant
`AKI_KDIGO`), avec les 5 colonnes du papier : tous les patients, dysnatremie,
jamais dysnatremiques, hyponatremie, hypernatremie.


In [5]:

CRF_t1 = CRF.copy()

CRF_t1["dysnatremia"]  = CRF_t1["Dysnatremia"].astype(bool)
CRF_t1["hyponatremia"] = CRF_t1["Hyponatremia"].astype(bool)
CRF_t1["hypernatremia"] = CRF_t1["Hypernatremia"].astype(bool)
CRF_t1["prop_hyponatremia"]  = CRF_t1["Hyponatremia"]
CRF_t1["prop_hypernatremia"] = CRF_t1["Hypernatremia"]

variables_categorisees_t1 = [
    'Sex', 'smoke', 'OH', 'AVC', 'hypertension', 'mechanical_ventilation_D1',
    'ards', 'Choc.hemo', 'Escarrotomie', 'infection', 'sep.shock',
    'antico.curative', 'bleed.dig', 'ECMO', 'ischemic.cardiac_disease',
    'congestive.heart.failure', 'peripheral.vasc.disease', 'renal.disease',
    'cancer', 'diabete', 'death', 'RRT',
    'MAKE28', 'MAKE90', 'electric.burn', 'chemical.burn', 'thermal.burn',
    'inhalation', 'hydrocobalamine', 'AKI_KDIGO', 'initial.cardiac.arrest',
    'pulmonary.disease', 'ObÃ©sitÃ©', 'severe.burn', 'LATA', 'Lyse_greffe',
    'prop_hyponatremia', 'prop_hypernatremia'
]
missing_cat = [c for c in variables_categorisees_t1 if c not in CRF_t1.columns]
variables_categorisees_t1 = [c for c in variables_categorisees_t1 if c in CRF_t1.columns]

variables_continues_t1 = [
    'Age', 'Charlson', 'weight', 'height', 'IMC', 'TBSA', 'TBSAfull', 'ABSI', 'IGSII',
    'UBS', 'first_natremia_value', 'creat_baseline', 'median_urea_value',
    'KDIGO', 'creat_last', 'N.surgery', 'T.stay', 'Creat.entry',
    'median_glucose_value', 'eGFR_adm', 'osmo_calc',
    'DelaiPEC',   # ajout 2026-07-29 : delai brulure -> admission, en jours (= D.entry - D.burn)
]
missing_cont = [c for c in variables_continues_t1 if c not in CRF_t1.columns]
variables_continues_t1 = [c for c in variables_continues_t1 if c in CRF_t1.columns]

if missing_cat or missing_cont:
    print("Colonnes absentes du dataframe (ignorees) :", missing_cat + missing_cont)

filtres = {
    "1_Tous":                  CRF_t1.index,
    "2_Dysnatremia":           CRF_t1[CRF_t1["dysnatremia"]].index,
    "3_Jamais_dysnatremiques": CRF_t1[~CRF_t1["dysnatremia"]].index,
    "4_Hyponatremia":          CRF_t1[CRF_t1["hyponatremia"]].index,
    "5_Hypernatremia":         CRF_t1[CRF_t1["hypernatremia"]].index,
}

tables = {}
for nom, idx in filtres.items():
    t = TableOne(
        data=CRF_t1.loc[idx],
        columns=variables_categorisees_t1 + variables_continues_t1,
        categorical=variables_categorisees_t1,
        nonnormal=variables_continues_t1,
        pval=False,
        missing=True,
    )
    tables[nom] = t.tableone

table1_5cols = pd.concat(tables.values(), axis=1, keys=tables.keys())
print(table1_5cols.head(25))

table1_5cols.to_excel("outputs/Table1_5_colonnes_regenere.xlsx")
print("\nExporte : Table1_5_colonnes_regenere.xlsx")


# --- Ligne "Time from burn to admission" de la Table 1 (ajout 2026-07-29) -----------
# Calculee explicitement pour rester reproductible meme sans `tableone`.
# DelaiPEC = D.entry - D.burn, en JOURS entiers (dates sans heure -> pas d'unite horaire possible).
_dp = pd.to_numeric(CRF_t1['DelaiPEC'], errors='coerce')
_groups_t1 = {
    'All':            CRF_t1.index,
    'Dysnatremia':    CRF_t1.index[CRF_t1['dysnatremia']],
    'No Dysnatremia': CRF_t1.index[~CRF_t1['dysnatremia']],
    'Hyponatremia':   CRF_t1.index[CRF_t1['hyponatremia']],
    'Hypernatremia':  CRF_t1.index[CRF_t1['hypernatremia']],
}
print("\nTime from burn to admission, days, median [Q1-Q3] :")
_row_delay = {}
for _g, _idx in _groups_t1.items():
    _v = _dp.loc[_idx].dropna()
    _row_delay[_g] = f"{_v.median():.1f} [{_v.quantile(.25):.1f}-{_v.quantile(.75):.1f}]"
    print(f"  {_g:15s} n={len(_idx):5d} (dispo {len(_v):5d})  {_row_delay[_g]:16s}"
          f"  same-day {int((_v == 0).sum()):4d} ({100*(_v == 0).mean():4.1f}%)"
          f"  >7 d {int((_v > 7).sum()):4d} ({100*(_v > 7).mean():4.1f}%)")
_miss_delay = int(_dp.isna().sum())
print(f"  -> donnee manquante chez {_miss_delay}/{len(CRF_t1)} patients (date de brulure absente).")
print("  -> les non-dysnatremiques sont admis PLUS TARD (1.5 [0.0-11.0] j) que les dysnatremiques")
print("     (1.0 [0.0-4.0] j) : transferts tardifs de brulures peu etendues. Max 201 j (n=9 > 60 j).")


                                       1_Tous              2_Dysnatremia             3_Jamais_dysnatremiques             4_Hyponatremia             5_Hypernatremia            
                                      Missing      Overall       Missing     Overall                 Missing     Overall        Missing     Overall         Missing     Overall
n                                                     1758                       990                                 768                        888                         418
Sex, n (%)                       0              694 (39.5)                383 (38.7)                          311 (40.5)                 347 (39.1)                  151 (36.1)
                                 1             1064 (60.5)                607 (61.3)                          457 (59.5)                 541 (60.9)                  267 (63.9)
smoke, n (%)                     0             1156 (65.8)                610 (61.6)                          546 (71.1)

## 4. Figure 1 — Incidence des dysnatremies

Barres d'incidence (Both / Hypernatremia / Hyponatremia / Dysnatremia) et
diagramme de Venn hypo vs hyper, a partir de `figure_1_incidence...ipynb`.


In [6]:
# Figure 1 - Incidence of dysnatremia (R1-FIX 2026-09-30 : libelles de l'axe x qui se chevauchaient
# ("Hyponatremia" / "Dysnatremia (any)") -> figure plus large, police reduite, libelle sur deux lignes,
# et ordre logique des barres (any -> hypo -> hyper -> both).
total_patients = len(CRF)

plot_data = pd.Series({
    'Dysnatremia\n(any)':  CRF['Dysnatremia'].sum(),
    'Hyponatremia':         CRF['Hyponatremia'].sum(),
    'Hypernatremia':        CRF['Hypernatremia'].sum(),
    'Both':                 CRF['Both'].sum(),
})

fig, ax = plt.subplots(figsize=(11, 6))
palette = [sns.color_palette("pastel")[i] for i in (3, 2, 1, 0)]

bars = ax.bar(range(len(plot_data)), plot_data.values, color=palette, edgecolor='black', linewidth=1, width=0.7)
for bar in bars:
    bar.set_path_effects([
        patheffects.withSimplePatchShadow(offset=(1, -1), shadow_rgbFace='grey', alpha=0.3)
    ])

ax.set_xticks(range(len(plot_data)))
ax.set_xticklabels(plot_data.index, fontsize=15, fontweight='bold')
ax.set_ylabel("Number of patients", fontsize=18, fontweight='bold')
ax.set_xlabel("")
ax.tick_params(axis='y', labelsize=12)
for lab in ax.get_yticklabels(): lab.set_fontweight('bold')

max_val = plot_data.max()
ax.set_ylim(0, max_val * 1.15)
ax.set_xlim(-0.6, len(plot_data) - 0.4)

for i, val in enumerate(plot_data.values):
    pct = (val / total_patients) * 100
    ax.text(i, val + max_val * 0.02, f"{val} ({pct:.1f}%)", ha='center', va='bottom', fontsize=16, fontweight='bold')

sns.despine()
plt.tight_layout()
plt.savefig('figures/figure1_incidence.png', dpi=200, bbox_inches='tight')   # R1 : Figure 1 enregistree
plt.show()


In [7]:

n_hypo  = CRF['Hyponatremia'].sum()
n_hyper = CRF['Hypernatremia'].sum()
n_both  = CRF['Both'].sum()

only_hypo  = n_hypo - n_both
only_hyper = n_hyper - n_both

palette = sns.color_palette("pastel")
color_hypo  = palette[0]
color_hyper = palette[3]

plt.figure(figsize=(8, 8))

venn = venn2(subsets=(only_hypo, only_hyper, n_both), set_labels=('Hyponatremia', 'Hypernatremia'))

venn.get_patch_by_id('10').set_color(color_hypo)
venn.get_patch_by_id('01').set_color(color_hyper)
venn.get_patch_by_id('11').set_color(palette[2])

for patch in ['10', '01', '11']:
    venn.get_patch_by_id(patch).set_alpha(0.7)

for label_id, count in zip(['10', '01', '11'], [only_hypo, only_hyper, n_both]):
    label = venn.get_label_by_id(label_id)
    if label:
        pct = count / total_patients * 100
        label.set_text(f"{count}\n({pct:.1f}%)")
        label.set_fontsize(14)
        label.set_fontweight('bold')

plt.title("Overlap between Hyponatremia and Hypernatremia", fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Figure 2 — MAKE et mortalite selon le statut natremique

A partir de `figure_2_post_these.ipynb`, **standardisee** pour utiliser
`MAKE_at_discharge` (MAKE90 seul), comme le reste du papier (Table 1, Abstract,
Cox) — la figure originale utilisait `MAKE28 OU MAKE90`, corrige ici par
decision du 2026-07-07 (voir Section 2).


In [8]:

print("MAKE utilise pour cette figure : MAKE_at_discharge (MAKE90), n =", CRF['MAKE_at_discharge'].sum())

def plot_make_rate(df, group_col, ax, title, make_col='MAKE_at_discharge'):
    contingency = pd.crosstab(df[group_col], df[make_col])
    make_rates = contingency.apply(lambda x: x.get(1, 0) / x.sum() * 100, axis=1)
    labels = make_rates.index.map({0: 'No', 1: 'Yes'})
    pastel_palette = sns.color_palette("pastel")[:2]
    bars = ax.bar(labels, make_rates.values, color=pastel_palette, edgecolor='black')

    col_labels = {"Dysnatremia": "Dysnatremia", "Hyponatremia": "Hyponatremia", "Hypernatremia": "Hypernatremia"}
    ax.set_title(title, fontsize=22, fontweight='bold')
    ax.set_xlabel(col_labels.get(group_col, group_col), fontsize=22, fontweight='bold')
    ax.set_ylabel("MAKE at discharge (%)", fontsize=22, fontweight='bold')
    ax.set_ylim(0, max(make_rates.values) * 1.4)
    ax.tick_params(axis='y', labelsize=16)
    ax.tick_params(axis='x', labelsize=20)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height, f"{height:.1f}%", ha='center', va='bottom', fontsize=22, fontweight='bold')

    chi2, p, _, _ = chi2_contingency(contingency)
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
    ax.text(0.5, 0.95, p_text, ha='center', va='top', fontsize=20, fontweight='bold', transform=ax.transAxes)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_make_rate(CRF, 'Dysnatremia', axes[0], 'a. Dysnatremia and MAKE at discharge')
plot_make_rate(CRF, 'Hyponatremia', axes[1], 'b. Hyponatremia and MAKE at discharge')
plot_make_rate(CRF, 'Hypernatremia', axes[2], 'c. Hypernatremia and MAKE at discharge')
plt.tight_layout()
plt.savefig("figures/figure2_make.png", dpi=200, bbox_inches='tight')
plt.show()


MAKE utilise pour cette figure : MAKE_at_discharge (MAKE90), n = 207


In [9]:

def plot_death_rate(df, group_col, ax, title):
    contingency = pd.crosstab(df[group_col], df['Death'])
    death_rates = contingency.apply(lambda x: x.get(1, 0) / x.sum() * 100, axis=1)
    labels = death_rates.index.map({0: 'No', 1: 'Yes'})
    pastel_palette = sns.color_palette("pastel")[:2]
    bars = ax.bar(labels, death_rates.values, color=pastel_palette, edgecolor='black')

    col_labels = {"Dysnatremia": "Dysnatremia", "Hyponatremia": "Hyponatremia", "Hypernatremia": "Hypernatremia"}
    ax.set_title(title, fontsize=22, fontweight='bold')
    ax.set_xlabel(col_labels.get(group_col, group_col), fontsize=22, fontweight='bold')
    ax.set_ylabel("Mortality (%)", fontsize=22, fontweight='bold')
    ax.set_ylim(0, max(death_rates.values) * 1.4)
    ax.tick_params(axis='y', labelsize=16)
    ax.tick_params(axis='x', labelsize=20)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height, f"{height:.1f}%", ha='center', va='bottom', fontsize=22, fontweight='bold')

    chi2, p, _, _ = chi2_contingency(contingency)
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
    ax.text(0.5, 0.95, p_text, ha='center', va='top', fontsize=20, fontweight='bold', transform=ax.transAxes)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_death_rate(CRF, 'Dysnatremia', axes[0], 'a. Dysnatremia and Mortality')
plot_death_rate(CRF, 'Hyponatremia', axes[1], 'b. Hyponatremia and Mortality')
plot_death_rate(CRF, 'Hypernatremia', axes[2], 'c. Hypernatremia and Mortality')
plt.tight_layout()
plt.savefig("figures/figure2_mortality.png", dpi=200, bbox_inches='tight')
plt.show()


## 6. Facteurs associes a l'hyponatremie / hypernatremie a l'admission

Regression logistique (a partir de `Facteurs associes Notebook.ipynb`).


In [10]:

CRF_fa = CRF.copy()
CRF_fa["hyponatremia_adm"]  = (CRF_fa["first_natremia_value"] < 135).astype(int)
CRF_fa["hypernatremia_adm"] = (CRF_fa["first_natremia_value"] > 145).astype(int)

categorical_vars = [
    'Sex', 'smoke', 'OH', 'AVC', 'hypertension',
    'ischemic.cardiac_disease', 'congestive.heart.failure', 'peripheral.vasc.disease',
    'renal.disease', 'cancer', 'diabete',            # R1-FIX 6 : 'death' retire (c'est un critere de jugement)
    'electric.burn', 'chemical.burn',                # R1-FIX 6 : brulure thermique = reference
    'inhalation', 'hydrocobalamine', 'pulmonary.disease', 'severe.burn'
]
categorical_vars = [c for c in categorical_vars if c in CRF_fa.columns]
continuous_vars = [c for c in ['Age', 'Charlson', 'IMC'] if c in CRF_fa.columns]

def prepare_data(df, outcome):
    X = df[categorical_vars + continuous_vars].copy()
    X = pd.get_dummies(X, columns=categorical_vars, drop_first=True)
    X = X.apply(pd.to_numeric, errors='coerce')
    y = df[outcome]
    data = pd.concat([X, y], axis=1).dropna()
    X = data.drop(columns=[outcome])
    y = data[outcome]
    X = X.loc[:, X.nunique() > 1]
    return X.astype(float), y.astype(int)

def logistic_regression(df, outcome):
    X, y = prepare_data(df, outcome)
    X = sm.add_constant(X)                           # R1-FIX 6 : constante
    model = sm.Logit(y, X)
    result = model.fit(method="lbfgs", disp=0, maxiter=20000, pgtol=1e-10, factr=10)   # R1-FIX 6 : tolerance stricte (la tolerance par defaut de lbfgs s arretait avant le maximum de vraisemblance)
    OR = np.exp(result.params)
    CI = result.conf_int()
    CI.columns = ["2.5%", "97.5%"]
    CI = np.exp(CI)
    summary = pd.DataFrame({
        "Variable": result.params.index,
        "OR": OR,
        "CI 2.5%": CI["2.5%"],
        "CI 97.5%": CI["97.5%"],
        "p-value": result.pvalues,
    })
    summary["OR"] = summary["OR"].round(3)
    summary["CI 2.5%"] = summary["CI 2.5%"].round(3)
    summary["CI 97.5%"] = summary["CI 97.5%"].round(3)
    summary["p-value"] = summary["p-value"].round(3)
    summary["Significant"] = summary["p-value"] < 0.05
    return summary.sort_values("p-value")

results_hypo_adm  = logistic_regression(CRF_fa, "hyponatremia_adm")
results_hyper_adm = logistic_regression(CRF_fa, "hypernatremia_adm")

print("===== HYPONATREMIE A L'ADMISSION =====")
print(results_hypo_adm.to_string(index=False))

print("\n===== HYPERNATREMIE A L'ADMISSION =====")
print(results_hyper_adm.to_string(index=False))


===== HYPONATREMIE A L'ADMISSION =====
                  Variable    OR  CI 2.5%  CI 97.5%  p-value  Significant
                     const 0.176    0.071     0.438    0.000         True
                 diabete_1 2.022    1.268     3.226    0.003         True
             severe.burn_1 1.510    1.103     2.067    0.010         True
         hydrocobalamine_1 0.253    0.085     0.748    0.013         True
           electric.burn_1 0.337    0.120     0.944    0.039         True
                       IMC 0.971    0.943     1.000    0.049         True
                      OH_1 1.447    0.994     2.108    0.054        False
                     AVC_1 0.411    0.152     1.112    0.080        False
              inhalation_1 0.658    0.402     1.075    0.095        False
                       Age 1.009    0.997     1.022    0.135        False
                  cancer_1 1.653    0.831     3.287    0.152        False
           chemical.burn_1 0.563    0.220     1.440    0.230        False

## 7. Modele de Cox a covariables temps-dependantes (Table 2)

A partir de `Cox Regression Notebook.ipynb`. Evenement = premier de
(deces, RRT, MAKE28, MAKE90) ; covariables temps-dependantes : hyponatremie,
hypernatremie. **Modification du 2026-07-11, sur demande de Louis Boutin
suite au commentaire de F. Depret ("j'ai du mal a comprendre l'ajustement
sur AKI") : l'AKI est retire de l'ajustement.** AKI n'est plus utilise ni
comme covariable ni comme point de coupure temporel.


In [11]:

df_cox = CRF.copy()

date_cols = ['D.burn', 'D.death', 'D.RRT', 'first_natremia_below_135_date', 'first_natremia_above_145_date']
for col in date_cols:
    if col in df_cox.columns:
        df_cox[col] = pd.to_datetime(df_cox[col], errors='coerce')

time_dep_cols = ['first_natremia_below_135_date', 'first_natremia_above_145_date']
for col in time_dep_cols:
    delay = (df_cox[col] - df_cox['D.burn']).dt.days
    df_cox.loc[delay < 0, col] = pd.NaT

def compute_event(row):
    origin = row['D.burn']
    candidates = []
    if pd.notnull(row['D.death']):
        d = (row['D.death'] - origin).days
        if d > 0:
            candidates.append(d)
    if pd.notnull(row['D.RRT']):
        d = (row['D.RRT'] - origin).days
        if d > 0:
            candidates.append(d)
    if row.get('MAKE28', 0) == 1:
        candidates.append(28)
    if row.get('MAKE90', 0) == 1:
        candidates.append(90)
    candidates = [c for c in candidates if c <= 90]   # R1-FIX 7 : evenements > J90 censures a J90
    if len(candidates) == 0:
        return pd.Series({'time': 90, 'event': 0})
    return pd.Series({'time': min(candidates), 'event': 1})

df_cox[['time', 'event']] = df_cox.apply(compute_event, axis=1)

rows = []
for i, r in df_cox.iterrows():
    end = r['time']
    origin = r['D.burn']

    hypo_t  = (r['first_natremia_below_135_date'] - origin).days if pd.notnull(r['first_natremia_below_135_date']) else np.inf
    hyper_t = (r['first_natremia_above_145_date'] - origin).days if pd.notnull(r['first_natremia_above_145_date']) else np.inf

    # R1-FIX 7 : seuls les changements d'exposition survenant AVANT la fin de suivi creent une coupure
    cuts = sorted(set([0, end] + [t for t in (hypo_t, hyper_t) if 0 < t < end]))

    for j in range(len(cuts) - 1):
        t0 = cuts[j]
        t1 = cuts[j + 1]
        if t0 == t1:
            continue
        rows.append({
            'id': i,
            'start': t0,
            'stop': t1,
            'event': int(r['event'] if t1 == end else 0),
            'Age': r['Age'],
            'TBSA': r['TBSA'],
            'hyponatremia': int(t0 >= hypo_t),
            'hypernatremia': int(t0 >= hyper_t),
        })

df_long = pd.DataFrame(rows)

ctv = CoxTimeVaryingFitter()
ctv.fit(
    df_long[['id', 'start', 'stop', 'event', 'Age', 'TBSA', 'hyponatremia', 'hypernatremia']],
    id_col='id', start_col='start', stop_col='stop', event_col='event',
)

summary = ctv.summary
for var in ['hyponatremia', 'hypernatremia']:
    hr   = np.exp(summary.loc[var, 'coef'])
    low  = np.exp(summary.loc[var, 'coef lower 95%'])
    high = np.exp(summary.loc[var, 'coef upper 95%'])
    p    = summary.loc[var, 'p']
    print(f"\n{var}")
    print(f"aHR = {hr:.2f}  IC95% = [{low:.2f} ; {high:.2f}]  p = {p:.4f}")

print("\nResume complet")
print(summary)



hyponatremia
aHR = 1.15  IC95% = [0.85 ; 1.55]  p = 0.3768

hypernatremia
aHR = 4.03  IC95% = [2.92 ; 5.56]  p = 0.0000

Resume complet
                   coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  exp(coef) lower 95%  exp(coef) upper 95%  cmp to          z             p    -log2(p)
covariate                                                                                                                                                          
Age            0.044306   1.045302  0.003831        0.036796        0.051815             1.037482             1.053181     0.0  11.563744  6.290486e-31  100.326599
TBSA           0.034288   1.034883  0.002548        0.029295        0.039282             1.029728             1.040063     0.0  13.458116  2.758613e-41  134.735109
hyponatremia   0.136436   1.146181  0.154383       -0.166150        0.439021             0.846920             1.551188     0.0   0.883748  3.768321e-01    1.408006
hypernatremia  1.393322   4.028211  0.16435

## 8. Figure 3 — Natremie et AKI/KDIGO

A partir de `figure_3_post_these.ipynb`, **standardisee** pour utiliser
`AKI_KDIGO`/`AKI_stage` corriges (comme le reste du papier) — la figure
originale utilisait les champs bruts `IRA`/`KDIGO`, remplaces ici par
decision du 2026-07-07 (voir Sections 1bis et 2). Le stade KDIGO 2, absent a
cause du bug, est desormais represente.


In [12]:

# Figure 3 - natremie (max / min / mediane du sejour) selon l'AKI et le stade KDIGO
# R1 : figure redessinee (libelles lisibles, effectifs, p en sous-titre) ; memes donnees et memes tests.
print("AKI_KDIGO, n =", CRF['AKI_KDIGO'].fillna(0).astype(int).sum())
plt.rcParams["font.family"] = "DejaVu Sans"
PAL3 = ["#9DC3E6", "#F4B183", "#A9D18E"]

def _pval(data, x_col, y_col):
    vals = sorted(data[x_col].dropna().unique())
    groups = [data.loc[data[x_col] == v, y_col].dropna() for v in vals]
    if len(groups) == 2:
        p = mannwhitneyu(groups[0], groups[1], alternative='two-sided').pvalue
    else:
        p = kruskal(*groups).pvalue
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

def _box(ax, data, x_col, y_col, labels, ylab, title):
    vals = sorted(data[x_col].dropna().unique())
    groups = [data.loc[data[x_col] == v, y_col].dropna().values for v in vals]
    bp = ax.boxplot(groups, widths=0.55, patch_artist=True, showfliers=False,
                    medianprops=dict(color='black', lw=1.4), whiskerprops=dict(color='#555555'),
                    capprops=dict(color='#555555'), boxprops=dict(edgecolor='#555555'))
    for b, c in zip(bp['boxes'], PAL3): b.set_facecolor(c)
    for i, g in enumerate(groups, start=1):
        med = np.median(g); q3 = np.percentile(g, 75)
        ax.text(i, q3, f"{med:.1f}", ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_xticks(range(1, len(vals) + 1))
    ax.set_xticklabels([f"{l}\n(n={len(g)})" for l, g in zip(labels, groups)], fontsize=10)
    ax.set_ylabel(ylab, fontsize=11); ax.set_xlabel("")
    ax.set_title(f"{title}\n{_pval(data, x_col, y_col)}", fontsize=11.5, loc='left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='y', color='#E6E6E6', lw=.6); ax.set_axisbelow(True)

label_map = {'max_natremia_value': "Maximum natremia (mmol/L)",
             'min_natremia_value': "Minimum natremia (mmol/L)",
             'median_natremia_value': "Median natremia (mmol/L)"}
CRF_AKI_KDIGO = CRF[(CRF['AKI_KDIGO'] == 1) & (CRF['AKI_stage'].notna()) & (CRF['AKI_stage'] != 0)]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for j, (y_col, ttl) in enumerate([('max_natremia_value', 'Maximum natremia'),
                                  ('min_natremia_value', 'Minimum natremia'),
                                  ('median_natremia_value', 'Median natremia')]):
    _box(axes[0, j], CRF, 'AKI_KDIGO', y_col, ['No AKI', 'AKI'], label_map[y_col], f"{chr(97+j)}. {ttl} by AKI status")
    _box(axes[1, j], CRF_AKI_KDIGO, 'AKI_stage', y_col, ['KDIGO 1', 'KDIGO 2', 'KDIGO 3'], label_map[y_col], f"{chr(100+j)}. {ttl} by KDIGO stage")
# echelles lisibles (les valeurs extremes sont hors bornes des moustaches, non tracees)
for ax in axes[:, 0]: ax.set_ylim(128, 172)
for ax in axes[:, 1]: ax.set_ylim(110, 150)
for ax in axes[:, 2]: ax.set_ylim(128, 152)
plt.tight_layout()
plt.savefig("figures/figure3_aki.png", dpi=200, bbox_inches='tight')
plt.show()


AKI_KDIGO, n = 404


In [13]:

def bootstrap_ci_median(data, n_boot=5000, ci=95):
    data = np.array(data.dropna())
    if len(data) == 0:
        return np.nan, np.nan
    boot_medians = np.random.choice(data, size=(n_boot, len(data)), replace=True)
    boot_medians = np.median(boot_medians, axis=1)
    alpha = (100 - ci) / 2
    lower = np.percentile(boot_medians, alpha)
    upper = np.percentile(boot_medians, 100 - alpha)
    return lower, upper

analyses = [("AKI_KDIGO", "max_natremia_value"), ("AKI_KDIGO", "min_natremia_value"), ("AKI_KDIGO", "median_natremia_value")]

CRF_AKI_KDIGO = CRF[(CRF["AKI_KDIGO"] == 1) & (CRF["AKI_stage"].notna()) & (CRF["AKI_stage"] != 0)]
analyses_kdigo = [("AKI_stage", "max_natremia_value"), ("AKI_stage", "min_natremia_value"), ("AKI_stage", "median_natremia_value")]

results = []
for group_var, value_var in analyses:
    tmp = CRF[[group_var, value_var]].dropna()
    for level in sorted(tmp[group_var].unique()):
        values = tmp.loc[tmp[group_var] == level, value_var]
        median = np.median(values)
        ci_low, ci_high = bootstrap_ci_median(values)
        results.append({"Analysis": value_var, "Group": f"{group_var}={level}", "N": len(values),
                         "Median": round(median, 1), "IC95% lower": round(ci_low, 1), "IC95% upper": round(ci_high, 1)})
    levels = sorted(tmp[group_var].unique())
    if len(levels) == 2:
        g1 = tmp.loc[tmp[group_var] == levels[0], value_var]
        g2 = tmp.loc[tmp[group_var] == levels[1], value_var]
        _, p = mannwhitneyu(g1, g2, alternative="two-sided")
    else:
        groups = [tmp.loc[tmp[group_var] == lvl, value_var] for lvl in levels]
        _, p = kruskal(*groups)
    results.append({"Analysis": value_var, "Group": "P-value", "N": "", "Median": "", "IC95% lower": "", "IC95% upper": p})

for group_var, value_var in analyses_kdigo:
    tmp = CRF_AKI_KDIGO[[group_var, value_var]].dropna()
    for level in sorted(tmp[group_var].unique()):
        values = tmp.loc[tmp[group_var] == level, value_var]
        median = np.median(values)
        ci_low, ci_high = bootstrap_ci_median(values)
        results.append({"Analysis": value_var, "Group": f"{group_var}={level}", "N": len(values),
                         "Median": round(median, 1), "IC95% lower": round(ci_low, 1), "IC95% upper": round(ci_high, 1)})
    groups = [tmp.loc[tmp[group_var] == lvl, value_var] for lvl in sorted(tmp[group_var].unique())]
    _, p = kruskal(*groups)
    results.append({"Analysis": value_var, "Group": "P-value", "N": "", "Median": "", "IC95% lower": "", "IC95% upper": p})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


             Analysis       Group    N Median IC95% lower  IC95% upper
   max_natremia_value AKI_KDIGO=0 1326  141.0       141.0 1.410000e+02
   max_natremia_value AKI_KDIGO=1  403  148.2       147.0 1.490000e+02
   max_natremia_value     P-value                         6.830658e-77
   min_natremia_value AKI_KDIGO=0 1326  135.0       135.0 1.355000e+02
   min_natremia_value AKI_KDIGO=1  403  132.6       132.0 1.330000e+02
   min_natremia_value     P-value                         1.265575e-24
median_natremia_value AKI_KDIGO=0 1326  138.0       138.0 1.384000e+02
median_natremia_value AKI_KDIGO=1  403  140.0       139.5 1.401000e+02
median_natremia_value     P-value                         2.630484e-19
   max_natremia_value AKI_stage=1  178  146.0       144.0 1.473000e+02
   max_natremia_value AKI_stage=2   73  146.1       144.6 1.490000e+02
   max_natremia_value AKI_stage=3  152  151.0       149.5 1.517000e+02
   max_natremia_value     P-value                         1.563008e-09
   min

## 9. Supplementary Figure 2 — Distribution de severite de la natremie

A partir de `supplementary_distribution_natremies_post_these.ipynb`. **Version
nettoyee** : les comptages/pourcentages sont calcules dynamiquement a partir des
donnees (la version originale du notebook contenait une cellule avec des
valeurs recopiees a la main — elle n'est pas reprise ici).


In [14]:
# Supplementary Figure 2 - min/max natremia severity distribution (from notebook cell 24)
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt, seaborn as sns
# R1 : CRF deja charge (le recharger ici effacait les colonnes derivees, p. ex. MAKE_at_discharge)
def compute_and_plot(col,cats,title,colors,ax):
    data=pd.to_numeric(CRF[col],errors="coerce").dropna(); nv=len(CRF)   # R1-FIX 12 : pourcentages sur toute la cohorte (n=1758), comme l incidence
    labels=list(cats.keys()); counts=pd.Series({k:cond(data).sum() for k,cond in cats.items()})
    pct=counts/nv*100
    bars=ax.bar(labels,counts.values,color=colors,edgecolor="white",width=0.9)
    for b,p in zip(bars,pct.values):
        ax.text(b.get_x()+b.get_width()/2,b.get_height(),f"{p:.1f}%",ha="center",va="bottom",fontsize=13,fontweight="bold")
    ax.set_title(title,fontsize=16,fontweight="bold",pad=12)
    ax.set_ylabel("Number of Patients",fontsize=15,labelpad=8)
    ax.tick_params(axis='x',labelrotation=45,labelsize=11); ax.tick_params(axis='y',labelsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    print(title,"(N=%d)"%nv); [print(f"  {l}: {int(counts[l])} ({p:.1f}%)") for l,p in zip(labels,pct.values)]
hypo_colors=sns.color_palette("Blues",5)[1:]; hyper_colors=sns.color_palette("Reds",5)[1:]; normal="#e9e9e9"
cmin={"Severe hyponatremia (<125)":lambda d:d<125,"Moderate hyponatremia (125-129)":lambda d:(d>=125)&(d<130),
      "Mild hyponatremia (130-134)":lambda d:(d>=130)&(d<135),"Normal (135-145)":lambda d:(d>=135)&(d<=145),
      "Hypernatremia (>145)":lambda d:d>145}
colmin=[hypo_colors[2],hypo_colors[1],hypo_colors[0],normal,hyper_colors[0]]
cmax={"Hyponatremia (<135)":lambda d:d<135,"Normal (135-145)":lambda d:(d>=135)&(d<=145),
      "Mild hypernatremia (146-150)":lambda d:(d>145)&(d<=150),"Moderate hypernatremia (151-155)":lambda d:(d>150)&(d<=155),
      "Severe hypernatremia (>155)":lambda d:d>155}
colmax=[hypo_colors[1],normal,hyper_colors[0],hyper_colors[1],hyper_colors[2]]
plt.rcParams.update({"font.family":"DejaVu Serif"})
fig,axes=plt.subplots(1,2,figsize=(16,8))
compute_and_plot("min_natremia_value",cmin,"Minimum Natremia Distribution",colmin,axes[0])
compute_and_plot("max_natremia_value",cmax,"Maximum Natremia Distribution",colmax,axes[1])
plt.tight_layout(); plt.savefig("figures/suppfig2_natremia_distribution.png",dpi=160,bbox_inches="tight",facecolor="white")
print("saved")
plt.show()

Minimum Natremia Distribution (N=1758)
  Severe hyponatremia (<125): 39 (2.2%)
  Moderate hyponatremia (125-129): 220 (12.5%)
  Mild hyponatremia (130-134): 629 (35.8%)
  Normal (135-145): 838 (47.7%)
  Hypernatremia (>145): 3 (0.2%)
Maximum Natremia Distribution (N=1758)
  Hyponatremia (<135): 16 (0.9%)
  Normal (135-145): 1295 (73.7%)
  Mild hypernatremia (146-150): 223 (12.7%)
  Moderate hypernatremia (151-155): 124 (7.1%)
  Severe hypernatremia (>155): 71 (4.0%)
saved


## Supplementary Figure 3 — Onsets of first dysnatremia episodes

Day of first hypo-/hypernatremia since burn, split at 10 days. Reproduces the published figure with current data.

In [15]:
# Supplementary Figure 3 - Onsets of first dysnatremia episodes (day since burn)
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt
# R1 : CRF deja charge (le recharger ici effacait les colonnes derivees, p. ex. MAKE_at_discharge)
for c in ["D.burn","first_natremia_below_135_date","first_natremia_above_145_date"]:
    CRF[c]=pd.to_datetime(CRF[c],errors="coerce")
origin=CRF["D.burn"]
edges=list(range(0,61,5)); labels=[f"{e}-{e+4}" for e in edges[:-1]]+[">60"]
def onset_days(col):
    t=(CRF[col]-origin).dt.days; return t[t.notna()]
def binned(t):
    b=np.clip((t//5).astype(int),0,12); return [int((b==i).sum()) for i in range(13)]
plt.rcParams.update({"font.family":"DejaVu Serif"})
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,(name,col,color,txtc) in zip(axes,[
        ("hyponatremia","first_natremia_below_135_date",(0.70,0.85,0.93),"#1f4e79"),
        ("hypernatremia","first_natremia_above_145_date",(0.94,0.65,0.63),"#a11d1d")]):
    t=onset_days(col); tot=len(t); early=int((t<10).sum()); late=tot-early
    pe,pl=100*early/tot,100*late/tot
    counts=binned(t)
    ax.bar(range(13),counts,width=0.9,color=color,edgecolor="white")
    ax.axvline(1.5,ls="--",color="black",lw=1.3)
    ymax=max(counts)*1.05
    ax.text(0.75,ymax*0.85,f"{pe:.1f}%\n<10 days",ha="center",va="top",fontsize=13,fontweight="bold",color=txtc)
    ax.text(5.5,ymax*0.85,f"{pl:.1f}%\n≥10 days",ha="center",va="top",fontsize=13,fontweight="bold",color=txtc)
    thr="135" if name=="hyponatremia" else "145"; sign="<" if name=="hyponatremia" else ">"
    ax.set_title(f"Day of first {name} (Na {sign} {thr} mmol/L)",fontsize=13)
    ax.set_xlabel("Day interval since burn injury",fontsize=12)
    ax.set_ylabel("Number of patients",fontsize=12)
    ax.set_xticks(range(13)); ax.set_xticklabels(labels,rotation=45,fontsize=9,ha="right")
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    print(f"{name}: n={tot} <10d {early} ({pe:.1f}%) >=10d {late} ({pl:.1f}%)")
plt.tight_layout(); plt.savefig("figures/suppfig3_onsets.png",dpi=160,bbox_inches="tight",facecolor="white")
print("saved")
plt.show()

hyponatremia: n=888 <10d 575 (64.8%) >=10d 313 (35.2%)
hypernatremia: n=418 <10d 318 (76.1%) >=10d 100 (23.9%)


saved


## 10. Episodes d'hyponatremie — bilan urinaire (Supplementary Figure 4)

A partir de `hyponatremia_episodes.ipynb` (version finale avec effectifs
diuretiques n/total). Le flow chart affiche les valeurs calculees a la volee,
pas de texte recopie a la main.


In [16]:

total_episodes_hypo = len(CRF2)

mask_urinary_data = (CRF2["has_UNa"] == 1) | (CRF2["has_Uosm"] == 1)
df_urinary = CRF2.loc[mask_urinary_data].copy()
total_urinary = len(df_urinary)

df_urinary["mean_Uosm"] = df_urinary[["min_Uosm", "max_Uosm"]].mean(axis=1, skipna=True)

low_osm  = df_urinary[df_urinary["mean_Uosm"] < 100]
high_osm = df_urinary[df_urinary["mean_Uosm"] >= 100]
n_low_osm, n_high_osm = len(low_osm), len(high_osm)

high_osm_low_UNa  = high_osm[high_osm["first_UNa"] < 20]
high_osm_high_UNa = high_osm[high_osm["first_UNa"] >= 20]
n_high_osm_low_UNa, n_high_osm_high_UNa = len(high_osm_low_UNa), len(high_osm_high_UNa)

def diuretics_stats(df):
    total = len(df)
    if total == 0:
        return 0, 0, 0
    n_diur = (df["n_diur_before"] >= 1).sum()   # R1-FIX 8
    pct = 100 * n_diur / total
    return n_diur, total, pct

low_osm_diur_n, low_osm_total, pct_low_osm = diuretics_stats(low_osm)
high_osm_low_UNa_diur_n, high_osm_low_UNa_total, pct_high_osm_low_UNa = diuretics_stats(high_osm_low_UNa)
high_osm_high_UNa_diur_n, high_osm_high_UNa_total, pct_high_osm_high_UNa = diuretics_stats(high_osm_high_UNa)

print("Hyponatremia episodes total       :", total_episodes_hypo)
print("Urine data available              :", total_urinary)
print("  Mean Uosm < 100                 :", n_low_osm)
print("  Mean Uosm >= 100                :", n_high_osm)
print("    UNa < 20                      :", n_high_osm_low_UNa)
print("    UNa >= 20                     :", n_high_osm_high_UNa)
print()
print(f"Diuretics before episode, Uosm<100          : {low_osm_diur_n}/{low_osm_total} ({pct_low_osm:.1f}%)")
print(f"Diuretics before episode, Uosm>=100 & UNa<20 : {high_osm_low_UNa_diur_n}/{high_osm_low_UNa_total} ({pct_high_osm_low_UNa:.1f}%)")
print(f"Diuretics before episode, Uosm>=100 & UNa>=20: {high_osm_high_UNa_diur_n}/{high_osm_high_UNa_total} ({pct_high_osm_high_UNa:.1f}%)")


Hyponatremia episodes total       : 3274
Urine data available              : 2416
  Mean Uosm < 100                 : 1
  Mean Uosm >= 100                : 2397
    UNa < 20                      : 111
    UNa >= 20                     : 2286

Diuretics before episode, Uosm<100          : 0/1 (0.0%)
Diuretics before episode, Uosm>=100 & UNa<20 : 1/111 (0.9%)
Diuretics before episode, Uosm>=100 & UNa>=20: 235/2286 (10.3%)


In [17]:
# Supplementary Figure 4 - Urinary characteristics of hyponatremic episodes
# Reproducible generator. Reads outputs/hypo_episode_full.xlsx (current pipeline),
# recomputes every count/percentage shown, and renders the published-style flow chart.
import pandas as pd, matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

CRF2 = pd.read_excel("outputs/hypo_episode_full.xlsx")

total = len(CRF2)
mask = (CRF2["has_UNa"]==1) | (CRF2["has_Uosm"]==1)
du = CRF2.loc[mask].copy(); n_urine = len(du)
du["mean_Uosm"] = du[["min_Uosm","max_Uosm"]].mean(axis=1, skipna=True)
high = du[du["mean_Uosm"]>=100]; n_high = len(high)
hl = high[high["first_UNa"]<20]; hh = high[high["first_UNa"]>=20]
n_lo, n_hi = len(hl), len(hh)
pct_lo, pct_hi = 100*n_lo/n_high, 100*n_hi/n_high
def diur(df):
    t=len(df); n=int((df["n_diur_before"]>=1).sum()); return n, (100*n/t if t else 0.0)
d_lo_n, d_lo_p = diur(hl); d_hi_n, d_hi_p = diur(hh)

plt.rcParams["font.family"]="DejaVu Serif"
plt.rcParams["mathtext.fontset"]="dejavuserif"
FS=15.5; FS2=14.5

fig, ax = plt.subplots(figsize=(10,7.1))
ax.set_xlim(0,100); ax.set_ylim(0,100); ax.axis("off")

def box(cx, cy, w, h, text):
    ax.add_patch(Rectangle((cx-w/2, cy-h/2), w, h, fill=True, facecolor="white",
                           edgecolor="black", linewidth=1.6, zorder=2))
    ax.text(cx, cy, text, ha="center", va="center", fontsize=FS, zorder=3, linespacing=1.5)

def vline(x, y1, y2):
    ax.plot([x,x],[y1,y2], color="black", lw=1.4, zorder=1)
def hline(x1, x2, y):
    ax.plot([x1,x2],[y,y], color="black", lw=1.4, zorder=1)

def pill(cx, cy, w=5.2, h=2.6):
    cap = FancyBboxPatch((cx-w/2, cy-h/2), w, h,
          boxstyle=f"round,pad=0,rounding_size={h/2}",
          linewidth=1.4, edgecolor="black", facecolor="white", zorder=4)
    ax.add_patch(cap)
    dark = Rectangle((cx-w/2, cy-h/2), w/2, h, facecolor="black", edgecolor="none", zorder=5)
    ax.add_patch(dark); dark.set_clip_path(cap)
    ax.plot([cx,cx],[cy-h/2, cy+h/2], color="black", lw=1.2, zorder=6)

# boxes
box(50, 92, 42, 12, "Total hyponatremia episodes\nn = %d" % total)
box(50, 72, 36, 11, "Available urinary data\nn = %d" % n_urine)
box(50, 52, 40, 11, r"Mean $U_{osm}$ $\geq$100 mOsm/L" + "\nn = %d" % n_high)
box(26, 28, 36, 12, r"First $U_{Na}$ < 20 mmol/L" + "\nn = %d (%.1f%%)" % (n_lo, pct_lo))
box(74, 28, 36, 12, r"First $U_{Na}$ $\geq$ 20 mmol/L" + "\nn = %d (%.1f%%)" % (n_hi, pct_hi))

# connectors
vline(50, 86, 77.5)
vline(50, 66.5, 57.5)
vline(50, 46.5, 40)
hline(26, 74, 40)
vline(26, 40, 34)
vline(74, 40, 34)

# diuretics under each UNa box
pill(26-10, 17); ax.text(26-6.5, 17, "n = %d (%.1f%%)" % (d_lo_n, d_lo_p), ha="left", va="center", fontsize=FS2)
pill(74-10, 17); ax.text(74-6.5, 17, "n = %d (%.1f%%)" % (d_hi_n, d_hi_p), ha="left", va="center", fontsize=FS2)

# legend
pill(6, 5); ax.text(10, 5, "Diuretics before episode,  n (%)", ha="left", va="center", fontsize=FS2)

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
out = "figures/suppfig4_hyponatremia_flowchart.png"
plt.savefig(out, dpi=200, bbox_inches="tight", facecolor="white")
from PIL import Image
print("numbers:", total, n_urine, n_high, n_lo, pct_lo, n_hi, pct_hi, "| diur", d_lo_n, d_lo_p, d_hi_n, d_hi_p)
print("saved:", out, Image.open(out).size)
plt.show()

numbers: 3274 2416 2397 111 4.630788485607009 2286 95.36921151439299 | diur 1 0.9009009009009009 235 10.279965004374453
saved: figures/suppfig4_hyponatremia_flowchart.png (2000, 1431)


## 11. Episodes d'hypernatremie — duree et remplissage vasculaire (Supplementary Figures 5-6)

A partir de `hypernatremia_episodes.ipynb` (version finale annotee).


In [18]:
# Supplementary Figure 5 - Distribution of hypernatremia episode duration
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
CRF3=pd.read_excel("outputs/hyper_episode_full.xlsx")
tot=len(CRF3); d=CRF3["n_days_episode"]
n_diur=int((CRF3["n_diur_before"]>=1).sum()); pct_diur=100*n_diur/tot
days=list(range(1,9)); counts=[int((d==k).sum()) for k in days]
plt.rcParams.update({"font.family":"DejaVu Serif"})
fig=plt.figure(figsize=(10.4,3.8))
ax=fig.add_axes([0.065,0.17,0.47,0.60])
ax.bar(days,counts,width=0.85,color=(0.6784,0.7843,0.9098),edgecolor="none")
ax.set_title("Distribution of hypernatremia episode duration",fontsize=13,fontweight="bold",pad=8)
ax.set_xlabel("Episode duration (days)",fontsize=13)
ax.set_ylabel("Number of episodes",fontsize=13)
ax.set_xticks(days); ax.set_xlim(0.4,8.6)
ax.tick_params(labelsize=11)
for s in ["top","right"]: ax.spines[s].set_visible(False)
# top-left count
fig.text(0.02,0.90,f"n = {tot} hypernatremia episodes",fontsize=15,ha="left",va="top")
# pill icon + diuretics text (right)
ic=mpimg.imread("../../Pipeline_analyse/icon_pill.png"); ih,iw=ic.shape[0],ic.shape[1]
figw,figh=fig.get_size_inches()
h_frac=0.20; w_frac=h_frac*figh/figw*(iw/ih)
iax=fig.add_axes([0.635,0.46,w_frac,h_frac]); iax.imshow(ic); iax.axis("off")
fig.text(0.635+w_frac+0.01,0.56,f"Diuretics before episode\nn = {n_diur}/{tot} ({pct_diur:.1f}%)",
         fontsize=14,ha="left",va="center",linespacing=1.5)
out="figures/suppfig5_hypernatremia_duration.png"
fig.savefig(out,dpi=160,facecolor="white")
print("saved",out,"counts",counts,"diur",n_diur,pct_diur)
plt.show()

saved figures/suppfig5_hypernatremia_duration.png counts [1708, 301, 94, 45, 29, 12, 4, 4] diur 337 15.276518585675431


In [19]:
# Supplementary Figure 6 - Vascular filling before hypernatremia episodes
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
CRF3=pd.read_excel("outputs/hyper_episode_full.xlsx")
tot=len(CRF3)
n_diur=int((CRF3["n_diur_before"]>=1).sum()); pct_diur=100*n_diur/tot
# R1-FIX 10 : 4 valeurs negatives (erreurs de saisie) traitees comme manquantes
fill=CRF3["mean_in_ml_before"].dropna(); fill=fill[fill>=0]; n_fill=len(fill); pct_fill=100*n_fill/tot
L=fill/1000
bins=[-1,1,2,3,4,5,6,L.max()+1]; labels=["<1 L","1-2 L","2-3 L","3-4 L","4-5 L","5-6 L",">=6 L"]
cat=pd.cut(L,bins=bins,labels=labels); c=cat.value_counts().sort_index()
counts=[int(c[l]) for l in labels]; pcts=[100*x/n_fill for x in counts]
plt.rcParams.update({"font.family":"DejaVu Serif"})
fig=plt.figure(figsize=(8.3,6.6)); figw,figh=fig.get_size_inches()
def place_icon(path,cx,cy,hfrac):
    ic=mpimg.imread(path); ih,iw=ic.shape[0],ic.shape[1]
    wfrac=hfrac*figh/figw*(iw/ih)
    a=fig.add_axes([cx-wfrac/2,cy-hfrac/2,wfrac,hfrac]); a.imshow(ic); a.axis("off")
# header: bag icon + text ; pill icon + text
place_icon("../../Pipeline_analyse/icon_bag.png",0.070,0.88,0.15)
fig.text(0.140,0.88,f"n = {n_fill} / {tot} ({pct_fill:.1f}%)\nwith vascular filling data available",
         fontsize=13,ha="left",va="center",linespacing=1.5)
place_icon("../../Pipeline_analyse/icon_pill.png",0.605,0.885,0.115)
fig.text(0.645,0.88,f"Diuretics before episode\nn = {n_diur}/{tot} ({pct_diur:.1f}%)",
         fontsize=13,ha="left",va="center",linespacing=1.5)
fig.text(0.5,0.77,"Vascular filling before hypernatremia episodes",fontsize=15,fontweight="bold",ha="center")
ax=fig.add_axes([0.10,0.14,0.86,0.55]); x=np.arange(len(labels))
ax.bar(x,counts,width=0.72,color=(0.945,0.760,0.588),edgecolor="none")
for i,(n,p) in enumerate(zip(counts,pcts)):
    ax.text(i,n+max(counts)*0.01,f"{n}\n{p:.1f}%",ha="center",va="bottom",fontsize=12,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels,rotation=22,fontsize=12)
ax.set_xlabel("Vascular filling before episode (L)",fontsize=13.5)
ax.set_ylabel("Number of episodes",fontsize=13.5)
ax.tick_params(axis="y",labelsize=11); ax.set_ylim(0,max(counts)*1.16)
for s in ["top","right"]: ax.spines[s].set_visible(False)
out="figures/suppfig6_vascular_filling.png"
fig.savefig(out,dpi=160,facecolor="white")
print("saved",out,counts,[round(p,1) for p in pcts])
plt.show()

saved figures/suppfig6_vascular_filling.png [564, 279, 104, 66, 20, 15, 24] [52.6, 26.0, 9.7, 6.2, 1.9, 1.4, 2.2]


## 12. Supplementary Figure 8 — Natremie d'admission et MAKE

A partir de `supplementary_figures_post_these_Naadmission.ipynb`, **standardisee**
pour utiliser `MAKE_at_discharge` (MAKE90 seul), comme le reste du papier — la
figure originale utilisait `MAKE28 OU MAKE90`, corrige ici par decision du
2026-07-07 (voir Section 2).


In [20]:

CRF['MAKE_label'] = CRF['MAKE_at_discharge'].map({0: 'No', 1: 'Yes'})
x_order = ['No', 'Yes']

def add_pvalue_and_title(data, x, y, ax, title):
    group0 = data[data[x] == 0][y].dropna()
    group1 = data[data[x] == 1][y].dropna()
    if len(group0) > 0 and len(group1) > 0:
        stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
        p_str = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        signif = " *" if p < 0.05 else ""
        ax.set_title(f"{title}\n({p_str}{signif})", fontsize=18, fontweight='bold')

def add_medians_by_order(data, x_col, y_col, ax, order):
    for i, group in enumerate(order):
        median = data.loc[data[x_col] == group, y_col].median()
        if not pd.isna(median):
            ax.text(i, median + (ax.get_ylim()[1] * 0.01), f"{median:.1f}", ha='center', va='bottom', fontsize=16, fontweight='bold')

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(x='MAKE_label', y='first_natremia_value', data=CRF, palette='pastel', ax=ax, order=x_order)
add_pvalue_and_title(CRF, 'MAKE_at_discharge', 'first_natremia_value', ax, "Admission natremia and MAKE at discharge")
add_medians_by_order(CRF, 'MAKE_label', 'first_natremia_value', ax, order=x_order)
ax.set_xlabel("MAKE at discharge", fontsize=18, fontweight='bold')
ax.set_ylabel("Admission natremia", fontsize=18, fontweight='bold')
ax.tick_params(axis='y', labelsize=14)
ax.tick_params(axis='x', labelsize=16)
ax.grid(False)
plt.tight_layout()
plt.savefig("figures/suppfig8_naadmission_make.png", dpi=200, bbox_inches='tight')
plt.show()


## 13. Clustering non supervise (methode A : predicteurs seuls) + AUC de dysnatremie (Figures 4-5)

**Revision 2026-07-21.** Suite au commentaire d'un relecteur, le clustering est realise uniquement sur des variables **predictives** (demographie, comorbidites, gravite de la brulure, physiologie d'admission). Les **outcomes** (deces, MAKE90, AKI_KDIGO, RRT, duree de sejour, chirurgies, complications) et les variables d'**exposition a la dysnatremie** sont EXCLUS du clustering ; leur association avec les clusters est evaluee dans un second temps independant. Cela evite toute circularite : la separation pronostique et le gradient de dysnatremie observes entre clusters ne sont plus imposes par construction. KMeans reimplemente en numpy (init k-means++, n_init=10) pour rendre le notebook autonome (sans dependance sklearn).

Caracterisation obtenue (cf. cellule) : **C** = grand groupe de reference peu severe ; **A** = phenotype age/comorbide, a predominance hyponatremique ; **B** = phenotype jeune grand-brule, le plus severe, ou se concentrent l'hypernatremie et la charge cumulee de dysnatremie.

---

_Note historique (version anterieure, tous predicteurs+outcomes melanges) :_

## 13. Clustering non supervise + AUC cumulee par dysnatremie (Figures 4-5)

A partir de `Clustering Non supervise + AUC.ipynb`, **rerun et corrige** :
la version originale recopiait a la main les valeurs d'AUC de la Figure 5
(`auc_hypo = [7.2, 19.2, 3.7]`, etc.), qui avaient derive par rapport a une
version anterieure du KMeans. Ici, les deux figures sont generees
integralement a partir du clustering recalcule dans la cellule ci-dessous —
aucune valeur n'est recopiee a la main.

**Caracterisation clinique des clusters** (confirmee avec Louis et deja
appliquee dans le manuscrit) : le cluster **B** est le plus petit (n=89) mais
le plus severe (mortalite ~74%, AKI_KDIGO ~93%, MAKE90 ~89%) ; le cluster
**C** est le grand groupe de reference, peu severe (n=1290) ; le cluster **A**
est intermediaire (n=379).

**Ajoute le 2026-07-12 (commentaires Word #44, #47, #67) :**
- Le tableau de caracteristiques des clusters (ci-dessous) est **trie par
  frequence decroissante** (cluster le plus nombreux en premier), les
  etiquettes A/B/C elles-memes ne changent pas (deja utilisees dans le texte
  du manuscrit).
- Ajout de l'AUC de dysnatremie cumulee (= AUC hyponatremie + AUC
  hypernatremie) en plus des AUC hypo/hyper deja presentes.
- Ajout de l'exposition au furosemide (Lasilix) par cluster, a partir de
  `data/raw/Mdt/daily_loop.xlsx`. **Limite importante** : ce fichier ne
  couvre que 293/1758 patients (16.7% de la cohorte), et ces patients sont
  concentres sur les annees recentes (brulure moyenne en 2019 vs 2016.6 pour
  le reste de la cohorte) -- vraisemblablement un probleme de disponibilite
  du dossier de prescription electronique plutot qu'une vraie absence de
  furosemide chez les patients plus anciens. Le pourcentage de patients sous
  furosemide et la dose totale rapportes ci-dessous sont donc calcules
  **uniquement parmi les patients avec donnee disponible** et ne sont pas
  necessairement representatifs de la cohorte complete ; a interpreter avec
  prudence.


In [21]:
# ================= Clustering non supervise -- METHODE A (predicteurs seuls) =================
# Revision 2026-07-21 (Louis) : suite au commentaire d'un relecteur, le clustering est
# desormais realise UNIQUEMENT sur des variables PREDICTIVES (demographie, comorbidites,
# gravite de la brulure, physiologie d'admission). Les OUTCOMES (deces, MAKE90, AKI_KDIGO,
# RRT, duree de sejour, chirurgies, complications...) ET les variables d'EXPOSITION a la
# dysnatremie sont volontairement EXCLUS du clustering ; leur association avec les clusters
# est evaluee dans un second temps independant (cf. caracterisation ci-dessous et Fig 4-5).
# KMeans reimplemente en numpy pur (aucune dependance sklearn), init k-means++, n_init=10.

def _standardize(A):
    A = A.astype(float); m = A.mean(0); s = A.std(0); s[s == 0] = 1.0
    return (A - m) / s

def _kpp_init(X, k, rng):
    n = X.shape[0]; centers = [X[rng.randint(n)]]
    for _ in range(1, k):
        d = np.min(((X[:, None, :] - np.array(centers)[None, :, :]) ** 2).sum(2), axis=1)
        p = d / d.sum(); centers.append(X[rng.choice(n, p=p)])
    return np.array(centers)

def kmeans_np(X, k=3, n_init=10, max_iter=300, seed=42):
    rng = np.random.RandomState(seed); best_lab = None; best_in = np.inf
    for _ in range(n_init):
        C = _kpp_init(X, k, rng)
        for _ in range(max_iter):
            lab = ((X[:, None, :] - C[None, :, :]) ** 2).sum(2).argmin(1)
            newC = np.array([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(k)])
            if np.allclose(newC, C):
                C = newC; break
            C = newC
        inertia = ((X - C[lab]) ** 2).sum()
        if inertia < best_in:
            best_in = inertia; best_lab = lab
    return best_lab, best_in

CRF_cl = CRF.copy()
for col in ["max_natremia_value", "min_natremia_value"]:
    if col in CRF_cl.columns:
        CRF_cl[col] = pd.to_numeric(CRF_cl[col], errors="coerce")
CRF_cl["Hypernatremia"] = (CRF_cl["max_natremia_value"] > 145).astype(int)
CRF_cl["Hyponatremia"]  = (CRF_cl["min_natremia_value"] < 135).astype(int)
CRF_cl["Dysnatremia"]   = ((CRF_cl["Hypernatremia"] == 1) | (CRF_cl["Hyponatremia"] == 1)).astype(int)

# ---- Variables PREDICTIVES utilisees pour le clustering (methode A) ----
variables_categorisees_cl = [
    'Sex', 'smoke', 'OH', 'AVC', 'hypertension', 'mechanical_ventilation_D1',
    'antico.curative', 'ischemic.cardiac_disease', 'congestive.heart.failure',
    'peripheral.vasc.disease', 'renal.disease', 'cancer', 'diabete', 'electric.burn',
    'chemical.burn', 'thermal.burn', 'inhalation', 'hydrocobalamine', 'severe.burn',
    'initial.cardiac.arrest', 'pulmonary.disease', 'IRA_obstructive', 'Epilespie',
    'dementia', 'connective.tissue.disorder', 'peptic.ulcer', 'liver.disease',
    'viral.hepatitis', 'hemiplegia', 'metastase', 'leukemia', 'lymphoma', 'HIV', 'cirrhosis',
]
variables_continues_cl = [
    'Age', 'Charlson', 'weight', 'height', 'IMC', 'TBSA', 'TBSAfull', 'ABSI', 'IGSII',
    'UBS', 'first_natremia_value', 'creat_baseline', 'Creat.entry', 'TP.base', 'Pl.base',
    'DelaiPEC',   # R1-FIX 11 : 'Hapto.base' retiree (79 % de donnees manquantes)
]
# Variables retirees du clustering (conservees pour la CARACTERISATION en 2e temps) :
#   outcomes/complications : death, MAKE90, AKI_KDIGO, RRT, KDIGO, creat_last, creat_90,
#     T.stay, LATA, D.RRT, Lyse_greffe, N_lyse_greffe, n.infection, n.septic.shock, ards,
#     Choc.hemo, sep.shock, CIVD, rhabdomyolyse, Hemolyse, IHC, bleed.dig, ECMO,
#     ACR_hospitalier, Escarrotomie, infection, N.surgery, nbCGR/nbPFC/nbCPA, CGR/PFC/CPA,
#     J.surgery1, Excision, parage, avulsion, Other_surgery, nadirTP, Anemie
#   exposition dysnatremie : Hypernatremia, Hyponatremia, Dysnatremia,
#     AUC_na_above145, AUC_na_below135, median_natremia_value

features = [c for c in (variables_categorisees_cl + variables_continues_cl) if c in CRF_cl.columns]
missing = [c for c in (variables_categorisees_cl + variables_continues_cl) if c not in CRF_cl.columns]
if missing:
    print("Colonnes absentes (ignorees) :", missing)

df_feat = CRF_cl[features].copy()
replace_map = {'??': np.nan, 'oui': 1, 'non': 0, 'OUI': 1, 'NON': 0}
for col in variables_categorisees_cl:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].replace(replace_map)
        df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce').fillna(0).astype(int)
for col in variables_continues_cl:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')
        df_feat[col] = df_feat[col].fillna(df_feat[col].median())
df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)
df_feat = df_feat.fillna(df_feat.median(numeric_only=True))

X_scaled = _standardize(df_feat.values)
labels, _inertia = kmeans_np(X_scaled, k=3, n_init=10, max_iter=300, seed=42)
CRF_cl['cluster'] = labels

# ---- Etiquetage deterministe A/B/C (independant de la numerotation KMeans) ----
# C = cluster le plus nombreux (reference, peu severe).
# Parmi les deux autres : B = TBSA mediane la plus elevee (grands brules, severe), A = l'autre (age/comorbide).
_sizes = CRF_cl.groupby('cluster').size()
_ref = _sizes.idxmax()
_others = [c for c in _sizes.index if c != _ref]
_tbsa = {c: CRF_cl.loc[CRF_cl.cluster == c, 'TBSA'].median() for c in _others}
_sev = max(_others, key=lambda c: _tbsa[c])
_oth = [c for c in _others if c != _sev][0]
lab_map = {_ref: 'C', _sev: 'B', _oth: 'A'}
CRF_cl['cluster_label'] = CRF_cl['cluster'].map(lab_map)
print("Correspondance numero KMeans -> etiquette :", lab_map)

CRF_cl['AUC_dysnatremia'] = CRF_cl['AUC_na_below135'] + CRF_cl['AUC_na_above145']

# --- Furosemide (Lasilix), commentaire Word #67 : couverture partielle (voir note markdown) ---
loop_raw = pd.read_excel('../../Pipeline_analyse/data/raw/Mdt/daily_loop.xlsx')
loop_raw['IPP2'] = pd.to_numeric(loop_raw['IPP'], errors='coerce')
loop_agg = loop_raw.groupby('IPP2', as_index=False).agg(furosemide_total_mg=('dose_mg', 'sum'))
CRF_cl = CRF_cl.merge(loop_agg, on='IPP2', how='left')
CRF_cl['has_furosemide_data'] = CRF_cl['furosemide_total_mg'].notna()
CRF_cl['furosemide_received'] = CRF_cl['furosemide_total_mg'].fillna(0) > 0
print(f"Couverture donnee furosemide : {CRF_cl['has_furosemide_data'].sum()}/{len(CRF_cl)} patients "
      f"({100*CRF_cl['has_furosemide_data'].mean():.1f}%)")

cluster_summary = CRF_cl.groupby('cluster_label').agg(
    n_patients=('cluster_label', 'count'),
    Age=('Age', 'median'), TBSA=('TBSA', 'median'), Charlson=('Charlson', 'median'),
    IGSII=('IGSII', 'median'), T_stay=('T.stay', 'median'), ABSI=('ABSI', 'median'),
    Days_Na_below_135=('AUC_na_below135', 'mean'), Days_Na_above_145=('AUC_na_above145', 'mean'),
    AUC_dysnatremia=('AUC_dysnatremia', 'mean'),
    Hyponatremia_pct=('Hyponatremia', 'mean'), Hypernatremia_pct=('Hypernatremia', 'mean'),
    Dysnatremia_pct=('Dysnatremia', 'mean'), AKI_KDIGO_pct=('AKI_KDIGO', 'mean'),
    Mortality_pct=('death', 'mean'), MAKE90_pct=('MAKE90', 'mean'),
    Ventilated_pct=('mechanical_ventilation_D1', 'mean'),
    Ventilated_n=('mechanical_ventilation_D1', 'sum'),
).round(2)
for col in ['Hyponatremia_pct', 'Hypernatremia_pct', 'Dysnatremia_pct', 'AKI_KDIGO_pct',
            'Mortality_pct', 'MAKE90_pct', 'Ventilated_pct']:
    cluster_summary[col] = (cluster_summary[col] * 100).round(1)

# furosemide : parmi les patients avec donnee disponible uniquement
furo = CRF_cl[CRF_cl['has_furosemide_data']].groupby('cluster_label').agg(
    furosemide_n_with_data=('has_furosemide_data', 'sum'),
    furosemide_pct_received=('furosemide_received', 'mean'),
)
furo['furosemide_pct_received'] = (furo['furosemide_pct_received'] * 100).round(1)
furo_dose = (CRF_cl[CRF_cl['furosemide_received']].groupby('cluster_label')['furosemide_total_mg']
             .median().round(0).rename('furosemide_median_mg_among_receivers'))
furo = furo.join(furo_dose)
cluster_summary = cluster_summary.join(furo)

cluster_summary = cluster_summary.sort_values('n_patients', ascending=False)
print(cluster_summary.to_string())


Correspondance numero KMeans -> etiquette : {np.int64(1): 'C', 2: 'B', 0: 'A'}


Couverture donnee furosemide : 293/1758 patients (16.7%)
               n_patients    Age  TBSA  Charlson  IGSII  T_stay  ABSI  Days_Na_below_135  Days_Na_above_145  AUC_dysnatremia  Hyponatremia_pct  Hypernatremia_pct  Dysnatremia_pct  AKI_KDIGO_pct  Mortality_pct  MAKE90_pct  Ventilated_pct  Ventilated_n  furosemide_n_with_data  furosemide_pct_received  furosemide_median_mg_among_receivers
cluster_label                                                                                                                                                                                                                                                                                                                                    
C                    1161  40.87   9.0       0.0   15.0    11.0   5.0               5.47               0.66             6.13              43.0                9.0             45.0            8.0            1.0         2.0            11.0         123.0                 

### 13a-bis. Tableau de caracterisation des clusters — version publication (median [Q1-Q3] + n/N)

**Ajout 2026-07-29 (Louis).** `cluster_summary` (cellule ci-dessus) ne fournit que des
**medianes nues** et des **pourcentages nus**, ce qui est insuffisant pour la publication.
Cette cellule reconstruit le meme tableau avec :

* **variables continues** : `median [Q1-Q3]` + le **n reellement disponible** (note de bas de tableau
  quand il est inferieur a la taille du cluster : IGSII, ABSI, AUC) ;
* **variables binaires** : `n/N (%)` avec **N = taille du cluster** ;
* **furosemide** : denominateur = patients ayant une donnee de prescription (couverture ~16.7 %).

**Trois incoherences corrigees par rapport au tableau precedemment redige** :

1. les lignes **AUC etaient des MOYENNES** (`Days_Na_below_135` = `.mean()`) presentees dans une
   colonne de medianes. Distributions tres asymetriques (SD >> moyenne) -> passage en mediane [Q1-Q3] ;
2. la ligne **« MAKE at discharge »** correspondait en realite a **MAKE90** (`MAKE90_pct`) ;
3. la ligne **Mortality du cluster A** affichait **19 %**, qui est la valeur de MAKE90 ; la mortalite
   reelle du cluster A est **51/329 = 15.5 %**.

**Point de vigilance signale** (cf. cellule d'audit) : un patient du cluster A porte une valeur
`AUC_na_below135 = 2095.7 mmol/L.day` physiquement impossible (borne max = (135 - Na_min) x duree de
sejour = 307.5). A lui seul il fait passer la moyenne d'AUC hyponatremie du cluster A de 8.68 a 15.16
et inverse l'ordre A > B affiche en **Figure 5**. La mediane, elle, n'est pas affectee.


In [22]:
# === Tableau de caracterisation des clusters, version publication ===============
# median [Q1-Q3] pour les continues, n/N (%) pour les binaires. Requiert CRF_cl (cellule ci-dessus).
ORDER_CL = ['C', 'A', 'B']
N_CL = {c: int((CRF_cl['cluster_label'] == c).sum()) for c in ORDER_CL}
N_TOT_CL = len(CRF_cl)
_num = lambda s: pd.to_numeric(s, errors='coerce')

CONT_CL = [
    ('Age, years',                              'Age',             1),
    ('TBSA, %',                                 'TBSA',            1),
    ('Charlson Comorbidity Index',              'Charlson',        1),
    ('SAPS II',                                 'IGSII',           1),
    ('ABSI',                                    'ABSI',            1),
    ('ICU length of stay, days',                'T.stay',          1),
    ('AUC hyponatremia, mmol/L.day',            'AUC_na_below135', 2),
    ('AUC hypernatremia, mmol/L.day',           'AUC_na_above145', 2),
    ('AUC dysnatremia (cumulative), mmol/L.day','AUC_dysnatremia',  2),
]
BIN_CL = [
    ('Mechanical ventilation',  'mechanical_ventilation_D1'),
    ('Hyponatremia',            'Hyponatremia'),
    ('Hypernatremia',           'Hypernatremia'),
    ('Dysnatremia',             'Dysnatremia'),
    ('AKI (KDIGO)',             'AKI_KDIGO'),
    ('MAKE90',                  'MAKE90'),
    ('Mortality (in-hospital)', 'death'),
]

rows_pub, rows_raw_pub, avail_cl = [], [], {}
rows_pub.append(['N, n (% of cohort)'] + [f"{N_CL[c]} ({100*N_CL[c]/N_TOT_CL:.1f})" for c in ORDER_CL])

for lab, col, dec in CONT_CL:
    cells = []
    for c in ORDER_CL:
        v = _num(CRF_cl.loc[CRF_cl.cluster_label == c, col]).dropna()
        med, q1, q3 = v.median(), v.quantile(.25), v.quantile(.75)
        cells.append(f"{med:.{dec}f} [{q1:.{dec}f}-{q3:.{dec}f}]")
        rows_raw_pub.append(dict(variable=col, cluster=c, kind='continuous', n_available=len(v),
                                 N=N_CL[c], median=round(med, 4), q1=round(q1, 4), q3=round(q3, 4),
                                 mean=round(v.mean(), 4), sd=round(v.std(), 4)))
        if len(v) < N_CL[c]:
            avail_cl.setdefault(lab, []).append(f"cluster {c} n={len(v)}/{N_CL[c]}")
    rows_pub.append([f"{lab}, median [Q1-Q3]"] + cells)

for lab, col in BIN_CL:
    cells = []
    for c in ORDER_CL:
        s = _num(CRF_cl.loc[CRF_cl.cluster_label == c, col]); k = int((s == 1).sum())
        cells.append(f"{k}/{N_CL[c]} ({100*k/N_CL[c]:.1f})")
        rows_raw_pub.append(dict(variable=col, cluster=c, kind='binary', n=k, N=N_CL[c],
                                 n_available=int(s.notna().sum()), pct=round(100*k/N_CL[c], 1)))
    rows_pub.append([f"{lab}, n/N (%)"] + cells)

_cf, _cd = [], []
for c in ORDER_CL:
    sub = CRF_cl[(CRF_cl.cluster_label == c) & CRF_cl.has_furosemide_data]
    nd = len(sub); k = int(sub.furosemide_received.sum())
    _cf.append(f"{k}/{nd} ({100*k/nd:.1f})" if nd else "-")
    dose = _num(CRF_cl.loc[(CRF_cl.cluster_label == c) & CRF_cl.furosemide_received,
                           'furosemide_total_mg']).dropna()
    _cd.append(f"{dose.median():.0f} [{dose.quantile(.25):.0f}-{dose.quantile(.75):.0f}] (n={len(dose)})"
               if len(dose) else "-")
    rows_raw_pub.append(dict(variable='furosemide_received', cluster=c, kind='binary',
                             n=k, N=nd, n_available=nd, pct=round(100*k/nd, 1) if nd else None))
    rows_raw_pub.append(dict(variable='furosemide_total_mg', cluster=c, kind='continuous',
                             n_available=len(dose), N=nd, median=round(dose.median(), 1),
                             q1=round(dose.quantile(.25), 1), q3=round(dose.quantile(.75), 1),
                             mean=round(dose.mean(), 1), sd=round(dose.std(), 1)))
rows_pub.append(['Furosemide received, n/n with data (%)'] + _cf)
rows_pub.append(['Furosemide total dose among receivers, median [Q1-Q3] (mg)'] + _cd)

header_cl = ['Characteristic'] + [f"Cluster {c} (n={N_CL[c]})" for c in ORDER_CL]
table_clusters_pub = pd.DataFrame(rows_pub, columns=header_cl)
table_clusters_raw = pd.DataFrame(rows_raw_pub)

_cov = CRF_cl.groupby('cluster_label')['has_furosemide_data'].sum()
footnotes_cl = [
    "Continuous variables are median [Q1-Q3]; categorical variables are n/N (%), with N = cluster size. "
    "Clusters were derived from predictor variables only; outcomes and dysnatremia exposure were not used to build them.",
]
for lab, lst in avail_cl.items():
    if lab.startswith('AUC'):
        continue
    footnotes_cl.append(f"* {lab}: available in " + ", ".join(lst) + "; median computed on available data.")
_auc_av = avail_cl.get('AUC hyponatremia, mmol/L.day', [])
footnotes_cl.append(
    "* AUC could only be computed in patients with >=2 natremia measurements ("
    + ", ".join(_auc_av) + "); missing values correspond to very short stays "
    "(median ICU length of stay 2 days).")
footnotes_cl.append(
    f"+ Loop-diuretic (furosemide) prescription data available for {int(CRF_cl.has_furosemide_data.sum())}/"
    f"{N_TOT_CL} patients ({100*CRF_cl.has_furosemide_data.mean():.1f}%): "
    + ", ".join(f"cluster {c} n={int(_cov.get(c, 0))}" for c in ORDER_CL)
    + ". Percentages use this subgroup as denominator.")

with pd.ExcelWriter('outputs/table_clusters_extended.xlsx') as _w:
    table_clusters_pub.to_excel(_w, sheet_name='Table_publication', index=False)
    table_clusters_raw.to_excel(_w, sheet_name='Raw_values', index=False)
    pd.DataFrame({'footnote': footnotes_cl}).to_excel(_w, sheet_name='Footnotes', index=False)

print(table_clusters_pub.to_string(index=False))
print()
for _f in footnotes_cl:
    print("  " + _f)


                                            Characteristic Cluster C (n=1161)   Cluster A (n=307)   Cluster B (n=290)
                                        N, n (% of cohort)        1161 (66.0)          307 (17.5)          290 (16.5)
                                Age, years, median [Q1-Q3]   40.9 [28.3-54.1]    72.9 [62.2-82.0]    50.1 [34.2-63.2]
                                   TBSA, %, median [Q1-Q3]     9.0 [4.0-15.0]     10.0 [5.0-16.8]    46.0 [35.0-64.8]
                Charlson Comorbidity Index, median [Q1-Q3]      0.0 [0.0-1.0]       5.0 [4.0-7.0]       1.0 [0.0-3.0]
                                   SAPS II, median [Q1-Q3]    15.0 [8.0-20.0]    27.0 [20.0-38.0]    36.0 [27.0-53.0]
                                      ABSI, median [Q1-Q3]      5.0 [4.0-6.0]       7.0 [6.0-8.0]     10.0 [8.0-12.0]
                  ICU length of stay, days, median [Q1-Q3]    11.0 [6.0-21.0]     15.0 [9.0-29.0]    44.5 [18.0-71.0]
              AUC hyponatremia, mmol/L.day, median [Q1-Q

In [23]:
# === Audit de plausibilite des AUC de natremie (ajout 2026-07-29) ===============
# Borne physique : AUC_hypo <= (135 - Na_min) x duree de sejour ; AUC_hyper <= (Na_max - 145) x duree.
_lo = pd.to_numeric(CRF_cl['AUC_na_below135'], errors='coerce')
_hi = pd.to_numeric(CRF_cl['AUC_na_above145'], errors='coerce')
_mn = pd.to_numeric(CRF_cl['min_natremia_value'], errors='coerce')
_mx = pd.to_numeric(CRF_cl['max_natremia_value'], errors='coerce')
_ts = pd.to_numeric(CRF_cl['T.stay'], errors='coerce')
_bound_lo = (135 - _mn).clip(lower=0) * _ts
_bound_hi = (_mx - 145).clip(lower=0) * _ts
# tolerance x2 : absorbe l'arrondi de T.stay en jours entiers
_flag = ((_lo > 2 * _bound_lo) & (_lo > 50)) | ((_hi > 2 * _bound_hi) & (_hi > 50))
print("Valeurs d'AUC physiquement impossibles :", int(_flag.sum()))
if _flag.any():
    print(CRF_cl.loc[_flag, ['IPP2', 'cluster_label', 'T.stay', 'min_natremia_value',
                             'max_natremia_value', 'AUC_na_below135', 'AUC_na_above145']].to_string(index=False))
    for _c in ['C', 'A', 'B']:
        _v = _lo[(CRF_cl.cluster_label == _c)].dropna()
        _v2 = _lo[(CRF_cl.cluster_label == _c) & (~_flag)].dropna()
        print(f"  cluster {_c} : mean AUC hypo avec outlier(s) = {_v.mean():6.2f} | sans = {_v2.mean():6.2f}")
    print("  -> la MEDIANE n'est pas affectee ; la Figure 5 (qui affiche des MOYENNES) l'est.")
print()
print("Couverture AUC :", int(_lo.notna().sum()), "/", len(CRF_cl),
      "; patients sans AUC mais avec natremie mesuree :",
      int((_lo.isna() & _mn.notna()).sum()))


Valeurs d'AUC physiquement impossibles : 0

Couverture AUC : 1545 / 1758 ; patients sans AUC mais avec natremie mesuree : 184


In [24]:

# Figure 4 — cercles descriptifs des clusters
colors = ['#AEC6CF', '#FFB347', '#B39EB5']
positions = [(1, 1), (3, 2), (5, 1)]

size_mapping = cluster_summary['n_patients'].rank(method='min').astype(int)
radius_map = {1: 1.2, 2: 1.6, 3: 2.0}
circle_sizes = size_mapping.map(radius_map)

fig, ax = plt.subplots(figsize=(12, 8))

for i, (label, row) in enumerate(cluster_summary.iterrows()):
    x, y = positions[i]
    radius = circle_sizes[label]
    circle = plt.Circle((x, y), radius=radius, color=colors[i], alpha=0.7)
    ax.add_patch(circle)
    text = (
        f"Cluster {label}\n"
        f"n = {int(row['n_patients'])}\n"
        f"Ventilated: {int(row['Ventilated_n'])} ({row['Ventilated_pct']:.0f}%)\n\n"
        f"Age: {row['Age']} y\n"
        f"TBSA: {row['TBSA']} %\n"
        f"Charlson: {row['Charlson']}\n"
        f"IGSII: {row['IGSII']}\n"
        f"T.stay: {row['T_stay']} d\n"
        f"ABSI: {row['ABSI']:.2f}\n\n"
        f"Na <135 (AUC): {row['Days_Na_below_135']:.1f}\n"
        f"Na >145 (AUC): {row['Days_Na_above_145']:.1f}\n\n"
        f"Hyponatremia: {row['Hyponatremia_pct']:.0f}%\n"
        f"Hypernatremia: {row['Hypernatremia_pct']:.0f}%\n"
        f"Dysnatremia: {row['Dysnatremia_pct']:.0f}%\n\n"
        f"AKI_KDIGO: {row['AKI_KDIGO_pct']:.0f}%\n"
        f"Mortality: {row['Mortality_pct']:.0f}%\n"
        f"MAKE90: {row['MAKE90_pct']:.0f}%"
    )
    ax.text(x, y, text, ha='center', va='center', fontsize=10, weight='bold')

ax.set_xlim(0, 6)
ax.set_ylim(0, 3.5)
ax.set_aspect('equal')
ax.axis('off')
plt.tight_layout()
plt.savefig("figures/figure4_clusters.png", dpi=200, bbox_inches='tight')
plt.show()


In [25]:

# Figure 4 (ex-Figure 5) - exposition cumulee a la dysnatremie (AUC) par cluster
# R1 : boites a moustaches (mediane [Q1-Q3]) au lieu des moyennes, coherent avec Supp Table (clusters) et le texte.
plt.rcParams["font.family"] = "DejaVu Sans"
ORDER = ['C', 'A', 'B']
PAL3 = ["#9DC3E6", "#F4B183", "#A9D18E"]
panels = [('AUC_na_below135', 'a. Hyponatremia burden'),
          ('AUC_na_above145', 'b. Hypernatremia burden'),
          ('AUC_dysnatremia', 'c. Cumulative dysnatremia burden')]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
med_out = {}
for ax, (col, ttl) in zip(axes, panels):
    groups = [pd.to_numeric(CRF_cl.loc[CRF_cl.cluster_label == c, col], errors='coerce').dropna().values for c in ORDER]
    bp = ax.boxplot(groups, widths=0.55, patch_artist=True, showfliers=False,
                    medianprops=dict(color='black', lw=1.4), whiskerprops=dict(color='#555555'),
                    capprops=dict(color='#555555'), boxprops=dict(edgecolor='#555555'))
    for b, c in zip(bp['boxes'], PAL3): b.set_facecolor(c)
    for i, (g, c) in enumerate(zip(groups, ORDER), start=1):
        med, q1, q3 = np.median(g), np.percentile(g, 25), np.percentile(g, 75)
        med_out[(col, c)] = (round(med, 2), round(q1, 2), round(q3, 2), len(g))
        ax.text(i, q3, f"{med:.1f}", ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels([f"Cluster {c}\n(n={len(g)})" for c, g in zip(ORDER, groups)], fontsize=10)
    ax.set_ylabel("AUC outside 135-145 mmol/L (mmol/L.day)", fontsize=10.5)
    ax.set_title(ttl, fontsize=11.5, loc='left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='y', color='#E6E6E6', lw=.6); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("figures/figure5_auc.png", dpi=200, bbox_inches='tight')
plt.show()
for k, v in med_out.items(): print(k, "median [Q1-Q3] =", v[0], v[1:3], "n =", v[3])


('AUC_na_below135', 'C') median [Q1-Q3] = 0.04 (np.float64(0.0), np.float64(5.86)) n = 967
('AUC_na_below135', 'A') median [Q1-Q3] = 0.67 (np.float64(0.0), np.float64(7.43)) n = 297
('AUC_na_below135', 'B') median [Q1-Q3] = 3.18 (np.float64(0.0), np.float64(12.64)) n = 281
('AUC_na_above145', 'C') median [Q1-Q3] = 0.0 (np.float64(0.0), np.float64(0.0)) n = 967
('AUC_na_above145', 'A') median [Q1-Q3] = 0.0 (np.float64(0.0), np.float64(0.03)) n = 297
('AUC_na_above145', 'B') median [Q1-Q3] = 5.82 (np.float64(0.35), np.float64(14.45)) n = 281
('AUC_dysnatremia', 'C') median [Q1-Q3] = 0.14 (np.float64(0.0), np.float64(6.64)) n = 967
('AUC_dysnatremia', 'A') median [Q1-Q3] = 2.15 (np.float64(0.0), np.float64(11.77)) n = 297
('AUC_dysnatremia', 'B') median [Q1-Q3] = 15.73 (np.float64(3.97), np.float64(32.57)) n = 281


### 13b. Selection du nombre de clusters (Supplementary Figure 10)

Methode du coude (inertie intra-cluster) et score de silhouette, calcules sur les memes variables predictives standardisees (`X_scaled`). k=3 est retenu pour l'interpretabilite clinique : la silhouette favorise marginalement k=2, mais une solution a deux clusters fusionnerait les phenotypes age/hyponatremique et jeune-grand-brule/hypernatremique.

In [26]:
def _silhouette(Xs, lab):
    D = np.sqrt(((Xs[:, None, :] - Xs[None, :, :]) ** 2).sum(2))
    ks = np.unique(lab); s = np.zeros(len(Xs))
    for i in range(len(Xs)):
        same = lab == lab[i]; same[i] = False
        a = D[i, same].mean() if same.any() else 0.0
        b = min(D[i, lab == c].mean() for c in ks if c != lab[i])
        s[i] = (b - a) / max(a, b) if max(a, b) > 0 else 0.0
    return s.mean()

ks_e = list(range(1, 9))
inertias = [kmeans_np(X_scaled, k, n_init=5, seed=42)[1] for k in ks_e]
ks_s = list(range(2, 7))
sils = [_silhouette(X_scaled, kmeans_np(X_scaled, k, n_init=5, seed=42)[0]) for k in ks_s]
print("inertias   :", [round(v) for v in inertias])
print("silhouettes:", [round(v, 3) for v in sils])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].plot(ks_e, inertias, 'o-', color='#4C72B0', lw=2, ms=7)
ax[0].axvline(3, color='#C44E52', ls='--', lw=1.5, alpha=.8)
ax[0].set_xlabel('Number of clusters (k)', fontsize=12)
ax[0].set_ylabel('Within-cluster inertia', fontsize=12)
ax[0].set_title('(A) Elbow method', fontsize=13); ax[0].set_xticks(ks_e)
ax[1].plot(ks_s, sils, 'o-', color='#4C72B0', lw=2, ms=7)
ax[1].axvline(3, color='#C44E52', ls='--', lw=1.5, alpha=.8)
ax[1].set_xlabel('Number of clusters (k)', fontsize=12)
ax[1].set_ylabel('Mean silhouette score', fontsize=12)
ax[1].set_title('(B) Silhouette analysis', fontsize=13); ax[1].set_xticks(ks_s)
for k, sv in zip(ks_s, sils):
    ax[1].annotate(f'{sv:.3f}', (k, sv), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
for a in ax:
    for sp in ['top', 'right']:
        a.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('figures/suppfig10_cluster_selection.png', dpi=200, bbox_inches='tight')
plt.show()


inertias   : [87900, 80341, 76205, 74130, 72128, 69661, 68176, 66402]
silhouettes: [np.float64(0.21), np.float64(0.196), np.float64(0.189), np.float64(0.183), np.float64(0.111)]


### 13c. Analyse de sensibilite : clustering hierarchique de Ward (Supplementary Figure 11)

Ward (numpy, valide contre scipy) sur les memes predicteurs standardises. Confirme les 3 memes phenotypes et le meme gradient ; concordance individuelle moderee (ARI ~0.32), attendue pour un gradient continu.

In [27]:
# ================= Analyse de sensibilite : clustering hierarchique de Ward (Supplementary Figure 11) =================
# Ward (Lance-Williams, distance euclidienne) reimplemente en numpy pur (aucune dependance scipy).
# Valide au prealable contre scipy.cluster.hierarchy (distances de fusion identiques ~1e-15, labels ARI=1.0).
# Reutilise X_scaled et CRF_cl (etiquettes k-means A/B/C) de la cellule de clustering ci-dessus.
from matplotlib.collections import LineCollection
import matplotlib.patches as mp
from math import comb

def ward_numpy(Xw):
    m = Xw.shape[0]; D = np.sqrt(((Xw[:, None, :] - Xw[None, :, :]) ** 2).sum(2)); np.fill_diagonal(D, np.inf)
    size = np.ones(m); active = np.ones(m, bool); cid = list(range(m)); nextid = m; Z = []; Dw = D.copy()
    for _ in range(m - 1):
        idx = np.argmin(Dw); i, j = divmod(idx, Dw.shape[0]); dd = Dw[i, j]; ni, nj = size[i], size[j]
        Z.append([cid[i], cid[j], dd, ni + nj]); nk = size; T = ni + nj + nk
        with np.errstate(invalid='ignore'):
            new = np.sqrt(((ni + nk) * Dw[i] ** 2 + (nj + nk) * Dw[j] ** 2 - nk * dd ** 2) / T)
        new[~active] = np.inf; new[i] = np.inf; new[j] = np.inf
        Dw[i, :] = new; Dw[:, i] = new; Dw[j, :] = np.inf; Dw[:, j] = np.inf
        size[i] = ni + nj; active[j] = False; cid[i] = nextid; nextid += 1
    return np.array(Z)

def fcluster_maxclust(Z, m, k):
    parent = list(range(m))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    rep = {i: i for i in range(m)}
    for mm in range(len(Z) - (k - 1)):
        a, b = int(Z[mm, 0]), int(Z[mm, 1]); ra, rb = find(rep[a]), find(rep[b]); parent[rb] = ra; rep[m + mm] = ra
    roots = [find(i) for i in range(m)]; uniq = {r: idx for idx, r in enumerate(sorted(set(roots)))}
    return np.array([uniq[r] for r in roots])

def _ari(a, b):
    ct = pd.crosstab(a, b).values; sij = sum(comb(v, 2) for v in ct.flatten())
    ai = ct.sum(1); bj = ct.sum(0); N = ct.sum(); sa = sum(comb(v, 2) for v in ai); sb = sum(comb(v, 2) for v in bj)
    exp = sa * sb / comb(N, 2); mx = (sa + sb) / 2; return (sij - exp) / (mx - exp)

nrow = X_scaled.shape[0]
Zw = ward_numpy(X_scaled)
wl = fcluster_maxclust(Zw, nrow, 3)
_w = CRF_cl[['TBSA']].copy(); _w['ward_raw'] = wl; _w['km'] = CRF_cl['cluster_label'].values
_sz = _w.groupby('ward_raw').size(); _ref = _sz.idxmax(); _oth = [c for c in _sz.index if c != _ref]
_tb = {c: _w.TBSA[_w.ward_raw == c].median() for c in _oth}; _sev = max(_oth, key=lambda c: _tb[c]); _o = [c for c in _oth if c != _sev][0]
_wmap = {_ref: 'C', _sev: 'B', _o: 'A'}; _w['ward'] = _w['ward_raw'].map(_wmap)
leafcol = _w['ward'].tolist()
ari_val = _ari(_w['km'], _w['ward']); conc = (_w['km'] == _w['ward']).mean() * 100
print("ARI k-means vs Ward: %.3f  | overall agreement: %.1f%%" % (ari_val, conc))
print(pd.crosstab(_w['ward'], _w['km']).reindex(index=['C','A','B'], columns=['C','A','B']).to_string())

COL = {'C': '#4C72B0', 'A': '#DD8452', 'B': '#C44E52'}
def _childs(nid): mm = nid - nrow; return int(Zw[mm, 0]), int(Zw[mm, 1])
height = np.zeros(2 * nrow - 1)
for mm in range(nrow - 1): height[nrow + mm] = Zw[mm, 2]
root = 2 * nrow - 2
seq = []; stk = [root]
while stk:
    nid = stk.pop()
    if nid < nrow: seq.append(nid)
    else:
        l, r = _childs(nid); stk.append(l); stk.append(r)
seq = seq[::-1]
xpos = np.zeros(2 * nrow - 1)
for i, lf in enumerate(seq): xpos[lf] = i
repleaf = np.arange(2 * nrow - 1)
for mm in range(nrow - 1):
    nid = nrow + mm; l, r = _childs(nid); xpos[nid] = (xpos[l] + xpos[r]) / 2.0; repleaf[nid] = repleaf[l]
thr = (Zw[-3, 2] + Zw[-2, 2]) / 2.0
segs = []; cols = []
for mm in range(nrow - 1):
    nid = nrow + mm; hh = Zw[mm, 2]; l, r = _childs(nid); xl, xr = xpos[l], xpos[r]; hl, hr = height[l], height[r]
    cc = COL[leafcol[repleaf[nid]]] if hh <= thr else '#B0B0B0'
    segs += [[(xl, hl), (xl, hh)], [(xr, hr), (xr, hh)], [(xl, hh), (xr, hh)]]; cols += [cc, cc, cc]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].add_collection(LineCollection(segs, colors=cols, linewidths=0.7))
ax[0].axhline(thr, color='k', ls='--', lw=1.2, alpha=.7)
ax[0].set_xlim(-5, nrow + 5); ax[0].set_ylim(0, Zw[-1, 2] * 1.05); ax[0].set_xticks([])
ax[0].set_ylabel('Ward linkage distance', fontsize=12)
ax[0].set_title('(A) Ward hierarchical dendrogram (cut into 3 clusters)', fontsize=12)
ax[0].legend(handles=[mp.Patch(color=COL[k], label=f'Cluster {k}') for k in ['C','A','B']], frameon=False, fontsize=10, loc='upper right')
for sp in ['top','right']: ax[0].spines[sp].set_visible(False)
CT = pd.crosstab(_w['ward'], _w['km']).reindex(index=['C','A','B'], columns=['C','A','B']).fillna(0).astype(int)
ax[1].imshow(CT.values, cmap='Blues'); ax[1].set_xticks([0,1,2]); ax[1].set_xticklabels(['C','A','B'])
ax[1].set_yticks([0,1,2]); ax[1].set_yticklabels(['C','A','B'])
ax[1].set_xlabel('k-means cluster (primary)', fontsize=12); ax[1].set_ylabel('Ward cluster', fontsize=12)
ax[1].set_title('(B) Concordance k-means vs Ward', fontsize=12)
mxv = CT.values.max()
for i in range(3):
    for j in range(3):
        v = CT.values[i, j]; ax[1].text(j, i, str(v), ha='center', va='center', color='white' if v > mxv*0.5 else 'black', fontsize=13, weight='bold')
fig.suptitle('Sensitivity analysis: Ward hierarchical vs k-means - ARI = %.2f, overall agreement = %.0f%%' % (ari_val, conc), fontsize=12.5, y=1.01)
plt.tight_layout(); plt.savefig('figures/suppfig11_hierarchical_sensitivity.png', dpi=200, bbox_inches='tight'); plt.show()


ARI k-means vs Ward: 0.511  | overall agreement: 81.3%
km       C    A    B
ward                
C     1019   38   92
A      132  266   54
B       10    3  144


## 13bis. Delai de survenue : dysnatremie vs AKI (Table supplementaire + Figure)

**Ajoute le 2026-07-11, sur demande de Louis Boutin suite aux commentaires
repetes de F. Depret ("on ne rapporte pas le timing de survenue des episodes
d'hypernatremie par rapport aux episodes d'AKI").** Deux analyses :

1. Parmi les patients qui developpent a la fois AKI et une dysnatremie
   (hypo- ou hypernatremie), delai (jours) entre le premier episode de
   dysnatremie et la date de survenue de l'AKI (KDIGO), et proportion pour
   laquelle la dysnatremie precede / est concomitante / suit l'AKI.
2. Sur l'ensemble de la cohorte, courbes d'incidence cumulee (1-Kaplan-Meier)
   du delai (depuis la brulure) jusqu'au premier episode d'AKI, de premiere
   hypernatremie et de premiere hyponatremie, avec censure a la sortie
   (ou au deces si anterieur a la sortie).


In [28]:

from lifelines import KaplanMeierFitter

date_cols_timing = ['D.burn', 'D.discharge', 'D.death', 'AKI_date',
                     'first_natremia_above_145_date', 'first_natremia_below_135_date']
for col in date_cols_timing:
    CRF[col] = pd.to_datetime(CRF[col], errors='coerce')

origin = CRF['D.burn']

# --- 1. Delai dysnatremie vs AKI, parmi les patients avec les deux ---
both_hyper = CRF[CRF['AKI_date'].notna() & CRF['first_natremia_above_145_date'].notna()].copy()
both_hyper['delay'] = (both_hyper['first_natremia_above_145_date'] - both_hyper['AKI_date']).dt.days

both_hypo = CRF[CRF['AKI_date'].notna() & CRF['first_natremia_below_135_date'].notna()].copy()
both_hypo['delay'] = (both_hypo['first_natremia_below_135_date'] - both_hypo['AKI_date']).dt.days

def timing_summary(df, label):
    n = len(df)
    precedes = int((df['delay'] < 0).sum())
    same_day = int((df['delay'] == 0).sum())
    follows  = int((df['delay'] > 0).sum())
    med = df['delay'].median()
    q1, q3 = df['delay'].quantile([0.25, 0.75])
    print(f"--- {label} vs AKI (n={n} patients avec les deux) ---")
    print(f"  {label} precede l'AKI : {precedes} ({100*precedes/n:.1f}%)")
    print(f"  Meme jour             : {same_day} ({100*same_day/n:.1f}%)")
    print(f"  {label} suit l'AKI    : {follows} ({100*follows/n:.1f}%)")
    print(f"  Delai median [Q1-Q3]  : {med:.1f} [{q1:.1f} - {q3:.1f}] jours")
    print()
    return {'label': label, 'n': n, 'precedes': precedes, 'precedes_pct': 100*precedes/n,
            'same_day': same_day, 'same_day_pct': 100*same_day/n,
            'follows': follows, 'follows_pct': 100*follows/n,
            'median_delay': med, 'q1': q1, 'q3': q3}

timing_table = pd.DataFrame([
    timing_summary(both_hyper, 'Hypernatremia'),
    timing_summary(both_hypo, 'Hyponatremia'),
])
timing_table.to_excel("outputs/timing_dysnatremia_vs_aki_table.xlsx", index=False)
timing_table


--- Hypernatremia vs AKI (n=257 patients avec les deux) ---
  Hypernatremia precede l'AKI : 120 (46.7%)
  Meme jour             : 36 (14.0%)
  Hypernatremia suit l'AKI    : 101 (39.3%)
  Delai median [Q1-Q3]  : 0.0 [-5.0 - 2.0] jours

--- Hyponatremia vs AKI (n=290 patients avec les deux) ---
  Hyponatremia precede l'AKI : 129 (44.5%)
  Meme jour             : 46 (15.9%)
  Hyponatremia suit l'AKI    : 115 (39.7%)
  Delai median [Q1-Q3]  : 0.0 [-6.8 - 9.0] jours



,label,n,precedes,precedes_pct,same_day,same_day_pct,follows,follows_pct,median_delay,q1,q3
0,Hypernatremia,257,120,46.692607,36,14.007782,101,39.299611,0.0,-5.00,2.0
1,Hyponatremia,290,129,44.482759,46,15.862069,115,39.655172,0.0,-6.75,9.0


In [29]:

# --- 2. Courbes d'incidence cumulee (1-KM) : AKI vs hypernatremie vs hyponatremie ---
censor_date = CRF['D.discharge'].fillna(CRF['D.death'])
followup = (censor_date - origin).dt.days.clip(lower=0)
mask = followup.notna()

def build_event_time(event_date):
    t_event = (event_date - origin).dt.days
    has_event = t_event.notna() & (t_event >= 0)
    time = np.where(has_event, t_event, followup)
    event = np.where(has_event, 1, 0)
    return pd.Series(time), pd.Series(event)

t_aki, e_aki     = build_event_time(CRF['AKI_date'])
t_hyper, e_hyper = build_event_time(CRF['first_natremia_above_145_date'])
t_hypo, e_hypo   = build_event_time(CRF['first_natremia_below_135_date'])

kmf_aki   = KaplanMeierFitter().fit(t_aki[mask],   e_aki[mask],   label='AKI (KDIGO)')
kmf_hyper = KaplanMeierFitter().fit(t_hyper[mask], e_hyper[mask], label='Hypernatremia')
kmf_hypo  = KaplanMeierFitter().fit(t_hypo[mask],  e_hypo[mask],  label='Hyponatremia')

XLIM = 30  # jours -- couvre la mediane/Q3 des 3 evenements, effectifs a risque encore suffisants

fig, ax = plt.subplots(figsize=(9, 6))
colors = {'AKI (KDIGO)': '#7f7f7f', 'Hyponatremia': '#1f77b4', 'Hypernatremia': '#d62728'}
for kmf in [kmf_aki, kmf_hypo, kmf_hyper]:
    label = kmf._label
    (1 - kmf.survival_function_).plot(ax=ax, color=colors[label], linewidth=2.5)
    ci = kmf.confidence_interval_cumulative_density_
    ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color=colors[label], alpha=0.15, step=None)

ax.set_xlim(0, XLIM)
ax.set_ylim(0, None)
ax.set_xlabel("Days since burn", fontsize=14)
ax.set_ylabel("Cumulative incidence", fontsize=14)
ax.set_title("Time to first AKI, hyponatremia, and hypernatremia episode", fontsize=15)
ax.legend(frameon=False, fontsize=12)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig("figures/figure_timing_aki_dysnatremia.png", dpi=200, bbox_inches='tight')
plt.show()

for name, kmf in [('AKI', kmf_aki), ('Hyponatremia', kmf_hypo), ('Hypernatremia', kmf_hyper)]:
    ci7  = 1 - kmf.survival_function_at_times(7).values[0]
    ci14 = 1 - kmf.survival_function_at_times(14).values[0]
    ci30 = 1 - kmf.survival_function_at_times(30).values[0]
    print(f"{name}: incidence cumulee (1-KM) a J7={ci7:.3f}, J14={ci14:.3f}, J30={ci30:.3f}")


AKI: incidence cumulee (1-KM) a J7=0.148, J14=0.199, J30=0.271
Hyponatremia: incidence cumulee (1-KM) a J7=0.324, J14=0.432, J30=0.604
Hypernatremia: incidence cumulee (1-KM) a J7=0.180, J14=0.231, J30=0.287


### Delai jusqu'au premier episode d'hyponatremie (verification hypothese hemodilution)

**Ajoute le 2026-07-12, commentaires Word #62/#63/#64 (F. Depret, Louis).** La
Discussion suggerait initialement que l'hyponatremie refletait surtout un
mecanisme dilutionnel/iatrogene "especially in the first 48-72 hours". Louis a
demande de verifier cette affirmation sur les donnees et, a defaut de preuve,
de supprimer la phrase. Calcul ci-dessous (delai brut premier episode
d'hyponatremie - date de brulure, non censure/KM, contrairement a la section
precedente) : la mediane est de J11, et seulement 31% des episodes
surviennent dans les 72 premieres heures -- la phrase a donc ete retiree et
remplacee par ces chiffres.


In [30]:

delay_hypo_raw = (CRF['first_natremia_below_135_date'] - CRF['D.burn']).dt.days.dropna()
delay_hyper_raw = (CRF['first_natremia_above_145_date'] - CRF['D.burn']).dt.days.dropna()

print(f"Hyponatremie (n={len(delay_hypo_raw)}) : mediane={delay_hypo_raw.median():.0f}j, "
      f"Q1={delay_hypo_raw.quantile(0.25):.2f}j, Q3={delay_hypo_raw.quantile(0.75):.0f}j, "
      f"% <= 48h = {100*(delay_hypo_raw<=2).mean():.1f}%, % <= 72h = {100*(delay_hypo_raw<=3).mean():.1f}%")
print(f"Hypernatremie (n={len(delay_hyper_raw)}) : mediane={delay_hyper_raw.median():.0f}j, "
      f"Q1={delay_hyper_raw.quantile(0.25):.2f}j, Q3={delay_hyper_raw.quantile(0.75):.0f}j")


Hyponatremie (n=888) : mediane=4j, Q1=2.00j, Q3=14j, % <= 48h = 39.3%, % <= 72h = 48.0%
Hypernatremie (n=418) : mediane=5j, Q1=2.00j, Q3=9j


## 14. Figure 2 (v2) — Odds ratios ajustes de l'AKI et du MAKE par categorie de natremie

**Ajoute le 2026-07-12.** Remplace l'ancienne Figure 2 (barres de taux bruts de
MAKE par sous-groupe dysnatremie/hypo/hyper), sur inspiration de la Figure 2 de
Funk et al., Intensive Care Med 2010 (dot-plot d'odds ratios par categorie de
natremie), etendue ici a deux issues (AKI et MAKE at discharge) au lieu d'une
seule (mortalite).

Chaque patient est classe dans l'une de 7 categories de natremie, selon sa
valeur extreme (min pour l'hyponatremie, max pour l'hypernatremie) :
`severe_hypo` (<125), `moderate_hypo` (125-130), `mild_hypo` (130-135),
`normal` (135-145, categorie de reference), `mild_hyper` (145-150),
`moderate_hyper` (150-155), `severe_hyper` (>155). Les patients ayant eu a la
fois un episode hypo- et hypernatremique sont classes selon l'ecart absolu a
la normale le plus important (le plus severe des deux).

Deux regressions logistiques (AKI_KDIGO ~ categorie + Age + TBSA ; puis
MAKE_at_discharge ~ categorie + Age + TBSA) fournissent les odds ratios
ajustes vs la categorie normale de reference.

**Note de tracabilite :** les valeurs numeriques exactes rapportees dans le
texte du manuscrit (Resultats, paragraphe Figure 2) proviennent d'une
execution figee de ce meme calcul. Une re-execution peut produire des OR
legerement differents (±0.1-0.2 sur le point-estimate) selon la version exacte
des packages (statsmodels) — les intervalles de confiance et l'interpretation
(relation dose-effet en U, plus marquee pour l'hypernatremie) restent
inchanges.


In [31]:

def categorize_natremia(row):
    mn, mx = row['min_natremia_value'], row['max_natremia_value']
    if pd.isna(mn) and pd.isna(mx):
        return np.nan   # R1-FIX 9 : patients sans natremie exploitable non classes 'normal'
    hypo_dev = (135 - mn) if pd.notna(mn) and mn < 135 else -np.inf
    hyper_dev = (mx - 145) if pd.notna(mx) and mx > 145 else -np.inf
    if hypo_dev == -np.inf and hyper_dev == -np.inf:
        return 'normal'
    if hyper_dev > hypo_dev:
        if mx > 155: return 'severe_hyper'
        elif mx > 150: return 'moderate_hyper'
        else: return 'mild_hyper'
    else:
        if mn < 125: return 'severe_hypo'
        elif mn < 130: return 'moderate_hypo'
        else: return 'mild_hypo'

CRF['na_cat'] = CRF.apply(categorize_natremia, axis=1)
na_order = ['severe_hypo', 'moderate_hypo', 'mild_hypo', 'normal',
            'mild_hyper', 'moderate_hyper', 'severe_hyper']
na_labels = ['Na<125', '125≤Na<130', '130≤Na<135', '135≤Na≤145',
             '145<Na≤150', '150<Na≤155', 'Na>155']
CRF['na_cat'] = pd.Categorical(CRF['na_cat'], categories=na_order, ordered=True)
print(CRF['na_cat'].value_counts().reindex(na_order))


na_cat
severe_hypo        36
moderate_hypo     188
mild_hypo         478
normal            739
mild_hyper        115
moderate_hyper    105
severe_hyper       68
Name: count, dtype: int64


In [32]:

import statsmodels.api as sm

na_dummies = pd.get_dummies(CRF['na_cat'], prefix='na', drop_first=False)
na_dummies = na_dummies.drop(columns=['na_normal'])

def fit_adjusted_or(outcome_col):
    d = pd.concat([CRF[[outcome_col, 'Age', 'TBSA']], na_dummies], axis=1)[CRF['na_cat'].notna()].dropna()   # R1-FIX 9
    X = sm.add_constant(d[list(na_dummies.columns) + ['Age', 'TBSA']].astype(float))
    y = d[outcome_col].astype(float)
    model = sm.Logit(y, X).fit(disp=0)
    out = {}
    for cat in na_order:
        if cat == 'normal':
            out[cat] = (1.0, 1.0, 1.0)
            continue
        col = f'na_{cat}'
        coef = model.params[col]
        ci = model.conf_int().loc[col]
        out[cat] = (np.exp(coef), np.exp(ci[0]), np.exp(ci[1]))
    return out, model

aki_or, aki_model = fit_adjusted_or('AKI_KDIGO')
make_or, make_model = fit_adjusted_or('MAKE_at_discharge')

print("Adjusted OR of AKI vs normonatremia (age+TBSA-adjusted):")
for cat in na_order:
    orr, lo, hi = aki_or[cat]
    print(f"  {cat}: {orr:.1f} [{lo:.1f}-{hi:.1f}]")

print("\\nAdjusted OR of MAKE at discharge vs normonatremia (age+TBSA-adjusted):")
for cat in na_order:
    orr, lo, hi = make_or[cat]
    print(f"  {cat}: {orr:.1f} [{lo:.1f}-{hi:.1f}]")


Adjusted OR of AKI vs normonatremia (age+TBSA-adjusted):
  severe_hypo: 8.4 [3.9-18.4]
  moderate_hypo: 4.5 [2.8-7.2]
  mild_hypo: 2.9 [1.9-4.3]
  normal: 1.0 [1.0-1.0]
  mild_hyper: 10.7 [6.3-18.0]
  moderate_hyper: 18.3 [10.1-33.0]
  severe_hyper: 13.8 [7.0-27.0]
\nAdjusted OR of MAKE at discharge vs normonatremia (age+TBSA-adjusted):
  severe_hypo: 1.5 [0.6-4.3]
  moderate_hypo: 0.8 [0.4-1.7]
  mild_hypo: 0.9 [0.5-1.5]
  normal: 1.0 [1.0-1.0]
  mild_hyper: 2.4 [1.3-4.5]
  moderate_hyper: 2.7 [1.4-5.1]
  severe_hyper: 4.0 [2.0-7.9]


In [33]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
panel_specs = [('a. AKI', aki_or), ('b. MAKE at ICU discharge', make_or)]

for ax, (title, results) in zip(axes, panel_specs):
    for i, cat in enumerate(na_order):
        orr, lo, hi = results[cat]
        is_ref = (cat == 'normal')
        ax.errorbar(i, orr, yerr=[[orr - lo], [hi - orr]], fmt='o',
                    color='black', ecolor='black', capsize=4,
                    markerfacecolor='white' if is_ref else 'black',
                    markersize=9, markeredgewidth=1.5)
        if is_ref:
            ax.annotate('reference', (i, orr), textcoords="offset points",
                        xytext=(0, -22), ha='center', fontsize=10, style='italic')
    ax.axhline(1, color='gray', linewidth=0.8)
    ax.set_yscale('log')
    ax.set_xticks(range(len(na_order)))
    ax.set_xticklabels(na_labels, rotation=30, ha='right')
    ax.set_ylabel('Adjusted Odds Ratio (95% CI), log scale', fontsize=12, fontweight='bold')
    ax.set_xlabel('Serum sodium (mmol/L)', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=15, fontweight='bold', loc='left')

plt.tight_layout()
plt.savefig("figures/figure2_aki_make_v2.png", dpi=200, bbox_inches='tight')
plt.show()


## 15. Chiffres cités dans le texte du manuscrit (ajout R1)
Tous les effectifs, médianes et p-values repris dans le texte, avec leurs dénominateurs.

In [34]:
# === Chiffres cites dans le texte du manuscrit (section ajoutee en R1) ===
from scipy.stats import mannwhitneyu, chi2_contingency, kruskal
def mq(s, d=1):
    s = pd.to_numeric(s, errors='coerce').dropna(); return f"{s.median():.{d}f} [{s.quantile(.25):.{d}f}-{s.quantile(.75):.{d}f}]"
def npct(mask, denom=None):
    denom = len(mask) if denom is None else denom; return f"{int(mask.sum())}/{denom} ({100*mask.sum()/denom:.1f}%)"
def pchi(a, b):
    p = chi2_contingency(pd.crosstab(a, b))[1]; return "<0.001" if p < 0.001 else f"{p:.3f}"
def pmw(x, y):
    p = mannwhitneyu(pd.to_numeric(x, errors='coerce').dropna(), pd.to_numeric(y, errors='coerce').dropna())[1]; return "<0.001" if p < 0.001 else f"{p:.3f}"
N = len(CRF)
print("— Population —")
print("Men", npct(CRF.Sex == 1), "| Age", mq(CRF.Age), "| TBSA", mq(CRF.TBSA), "| full-thickness", mq(CRF.TBSAfull))
print("Inhalation", npct(CRF.inhalation == 1), "| CKD", npct(CRF['renal.disease'] == 1), "| KRT", npct(CRF.RRT == 1), "| MAKE", npct(CRF.MAKE_at_discharge == 1), "| death", npct(CRF.Death == 1), "| AKI", npct(CRF.AKI_KDIGO == 1))
print("— Incidence —")
print("Dysnatremia", npct(CRF.Dysnatremia == 1), "| both", npct(CRF.Both == 1), "| hypo", npct(CRF.Hyponatremia == 1), "| hyper", npct(CRF.Hypernatremia == 1))
fn = CRF.first_natremia_value; ha, Ha = fn < 135, fn > 145; nn = fn.notna() & ~ha & ~Ha
print("Admission natremia", mq(fn), "| hypo at admission", npct(ha), "| hyper at admission", npct(Ha))
dp = pd.to_numeric(CRF.DelaiPEC, errors='coerce')
print("Delay burn->admission: all", mq(dp), "| hypo adm", mq(dp[ha]), "p", pmw(dp[ha], dp[nn]), "| hyper adm", mq(dp[Ha]), "p", pmw(dp[Ha], dp[nn]), "| normo adm", mq(dp[nn]))
print("AUC hypo among hyponatremic", mq(CRF.loc[CRF.Hyponatremia == 1, 'AUC_na_below135']), "| AUC hyper among hypernatremic", mq(CRF.loc[CRF.Hypernatremia == 1, 'AUC_na_above145']))
print("Hypo episodes", len(CRF2), "| with urine", npct(((CRF2.has_UNa == 1) | (CRF2.has_Uosm == 1))))
print("Hyper episodes", len(CRF3), "| <=1 day", npct(CRF3.n_days_episode <= 1), "| diuretic before", npct(CRF3.n_diur_before >= 1), "| filling data", npct(CRF3.mean_in_ml_before >= 0), "| <1 L", npct(CRF3.mean_in_ml_before[CRF3.mean_in_ml_before >= 0] < 1000))
print("— Outcomes by group —")
for g in ['Dysnatremia', 'Hypernatremia', 'Hyponatremia']:
    for oc in ['MAKE_at_discharge', 'Death']:
        a, b = CRF[g] == 1, CRF[g] == 0
        print(f"{oc:18s} {g:14s} yes {npct(CRF.loc[a, oc] == 1)} vs no {npct(CRF.loc[b, oc] == 1)} p={pchi(CRF[g], CRF[oc])}")
print("— Natremia by AKI / KDIGO —")
for v in ['max_natremia_value', 'median_natremia_value', 'min_natremia_value']:
    a, b = CRF.loc[CRF.AKI_KDIGO == 1, v], CRF.loc[CRF.AKI_KDIGO == 0, v]
    st = [CRF.loc[CRF.AKI_stage == k, v].dropna() for k in (1, 2, 3)]; pk = kruskal(*st)[1]
    print(v, "AKI", mq(a), "vs no AKI", mq(b), "p", pmw(a, b), "| KDIGO1", mq(st[0]), "KDIGO2", mq(st[1]), "KDIGO3", mq(st[2]), "p", "<0.001" if pk < 0.001 else f"{pk:.3f}")
print("— Table 1 medians for discussion — TBSA hypo", mq(CRF.loc[CRF.Hyponatremia == 1, 'TBSA']), "| hyper", mq(CRF.loc[CRF.Hypernatremia == 1, 'TBSA']))


— Population —
Men 1064/1758 (60.5%) | Age 47.6 [31.5-62.3] | TBSA 11.0 [5.0-21.0] | full-thickness 3.5 [1.0-10.0]
Inhalation 248/1758 (14.1%) | CKD 50/1758 (2.8%) | KRT 103/1758 (5.9%) | MAKE 207/1758 (11.8%) | death 163/1758 (9.3%) | AKI 404/1758 (23.0%)
— Incidence —
Dysnatremia 990/1758 (56.3%) | both 316/1758 (18.0%) | hypo 888/1758 (50.5%) | hyper 418/1758 (23.8%)
Admission natremia 139.0 [136.4-141.0] | hypo at admission 249/1758 (14.2%) | hyper at admission 65/1758 (3.7%)
Delay burn->admission: all 1.0 [0.0-6.2] | hypo adm 2.0 [0.0-6.0] p <0.001 | hyper adm 0.0 [0.0-5.0] p 0.033 | normo adm 1.0 [0.0-7.0]
AUC hypo among hyponatremic 6.1 [1.4-15.3] | AUC hyper among hypernatremic 5.6 [0.9-13.8]
Hypo episodes 3274 | with urine 2416/3274 (73.8%)
Hyper episodes 2206 | <=1 day 1708/2206 (77.4%) | diuretic before 337/2206 (15.3%) | filling data 1072/2206 (48.6%) | <1 L 564/1072 (52.6%)
— Outcomes by group —
MAKE_at_discharge  Dysnatremia    yes 163/990 (16.5%) vs no 44/768 (5.7%) p=<0

## Fin du notebook

Toutes les analyses du papier ont ete regroupees ici avec des chemins
relatifs et des valeurs calculees dynamiquement. Les deux incoherences de
definition identifiees (IRA vs AKI_KDIGO, MAKE28-ou-90 vs MAKE90 seul) sont
signalees a chaque endroit concerne plutot que d'etre resolues silencieusement,
pour rester coherent avec l'objectif de traçabilite/coherence du dossier.
